In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

import matplotlib.pyplot as plt

path = "/kaggle/input/competitions/m5-forecasting-accuracy"

calendar = pd.read_csv(f"{path}/calendar.csv")
sales = pd.read_csv(f"{path}/sales_train_validation.csv")
prices = pd.read_csv(f"{path}/sell_prices.csv")

print("calendar:", calendar.shape)
print("sales:", sales.shape)
print("prices:", prices.shape)

/kaggle/input/competitions/m5-forecasting-accuracy/calendar.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sample_submission.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sell_prices.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_validation.csv
/kaggle/input/competitions/m5-forecasting-accuracy/sales_train_evaluation.csv
calendar: (1969, 14)
sales: (30490, 1919)
prices: (6841121, 4)


In [ ]:
sales.head()

In [ ]:

import os
import gc
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# 0. 文件路径与参数
# ============================================================
BASE = "/kaggle/input/competitions/m5-forecasting-accuracy/"
OUT = "/kaggle/working/"

N_SERIES = 5000
TRAIN_START = 1466
FIT_START = 1522
VALID_START = 1914
VALID_END = 1941

# ============================================================
# 1. 读取数据
# ============================================================
sales_val = pd.read_csv(BASE + "sales_train_validation.csv")
sales_eval = pd.read_csv(BASE + "sales_train_evaluation.csv")
calendar = pd.read_csv(BASE + "calendar.csv")
prices = pd.read_csv(BASE + "sell_prices.csv")
sample = pd.read_csv(BASE + "sample_submission.csv")

calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday"] = calendar["date"].dt.dayofweek
calendar["is_event"] = (
    calendar["event_name_1"].notna() |
    calendar["event_name_2"].notna()
).astype("int8")

calendar_map = calendar.set_index("d")
price_cols = ["store_id", "item_id", "wm_yr_wk", "sell_price"]

meta_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

# 类别编码规则统一使用完整数据，保证训练和预测一致
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
category_levels = {
    c: pd.Index(sales_eval[c].unique())
    for c in cat_cols
}

def add_category_codes(df):
    df = df.copy()
    for c in cat_cols:
        df[c + "_code"] = pd.Categorical(
            df[c], categories=category_levels[c]
        ).codes.astype("int32")
    return df

# ============================================================
# 2. 抽取训练样本，并转换成长表
# ============================================================
rng = np.random.RandomState(42)
n = min(N_SERIES, len(sales_eval))
sample_idx = rng.choice(len(sales_eval), n, replace=False)

all_day_cols = [
    f"d_{i}" for i in range(TRAIN_START, VALID_END + 1)
]

sampled = sales_eval.iloc[sample_idx][meta_cols + all_day_cols].copy()

long_df = sampled.melt(
    id_vars=meta_cols,
    value_vars=all_day_cols,
    var_name="d",
    value_name="sales"
)

long_df["day_num"] = long_df["d"].str[2:].astype("int16")

# 添加日历信息、售价
calendar_features = [
    "d", "wm_yr_wk", "weekday", "month", "year",
    "is_event", "snap_CA", "snap_TX", "snap_WI"
]

long_df = long_df.merge(
    calendar[calendar_features],
    on="d",
    how="left"
)

long_df = long_df.merge(
    prices[price_cols],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# SNAP 是按州定义的福利日标记
long_df["snap"] = np.select(
    [
        long_df["state_id"].eq("CA"),
        long_df["state_id"].eq("TX"),
        long_df["state_id"].eq("WI")
    ],
    [
        long_df["snap_CA"],
        long_df["snap_TX"],
        long_df["snap_WI"]
    ],
    default=0
).astype("int8")

long_df = add_category_codes(long_df)
long_df = long_df.sort_values(["id", "day_num"]).reset_index(drop=True)

# ============================================================
# 3. 建立预测特征
# 注意：滚动特征只读取目标日之前至少 28 天的销量
# ============================================================
g = long_df.groupby("id", sort=False)["sales"]

long_df["lag_28"] = g.shift(28)
long_df["lag_56"] = g.shift(56)

long_df["rolling_mean_7"] = (
    long_df.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(7).mean())
)

long_df["rolling_mean_28"] = (
    long_df.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(28).mean())
)


features = [
    "weekday", "month", "year",
    "is_event", "snap", "sell_price",
    "lag_28", "lag_56",
    "rolling_mean_7", "rolling_mean_28"
]


categorical_feature=[]

data = long_df.dropna(
    subset=features + ["sales"]
).copy()

# 训练样本不能使用验证期目标值
train_df = data[
    (data["day_num"] >= FIT_START) &
    (data["day_num"] < VALID_START)
]

valid_df = data[
    (data["day_num"] >= VALID_START) &
    (data["day_num"] <= VALID_END)
]

print("训练样本行数：", len(train_df))
print("验证样本行数：", len(valid_df))
print("使用的特征数：", len(features))

# ============================================================
# 4. 训练验证模型：先用历史数据预测连续 28 天
# ============================================================
params = dict(
    objective="tweedie",
    tweedie_variance_power=1.2,
    n_estimators=350,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=80,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_val = LGBMRegressor(**params)

model_val.fit(
    train_df[features],
    train_df["sales"],
    categorical_feature=[]
)

valid_pred = np.maximum(
    model_val.predict(valid_df[features]), 0
)

mae = mean_absolute_error(valid_df["sales"], valid_pred)
rmse = np.sqrt(
    mean_squared_error(valid_df["sales"], valid_pred)
)

print("\n========== 28 天验证结果 ==========")
print("MAE：", round(mae, 4))
print("RMSE：", round(rmse, 4))

# ============================================================
# 5. 预测函数：直接预测未来 28 天
# 每个预测日都使用目标日前 28 天及更早的真实销量特征
# 因此不需要把预测值递归塞回历史
# ============================================================
def forecast_direct(model, sales_df, history_day_cols, future_days):
    history = sales_df[history_day_cols].to_numpy(dtype=np.float32)
    n_series, n_history = history.shape

    meta = sales_df[meta_cols].copy()
    meta = add_category_codes(meta)

    outputs = []

    for i, d in enumerate(future_days):
        cal = calendar_map.loc[d]

        # 未来第 i 天对应的目标日索引是 n_history + i
        t = n_history + i

        future = meta.copy()
        future["d"] = d
        future["day_num"] = int(d[2:])

        future["lag_28"] = history[:, t - 28]
        future["lag_56"] = history[:, t - 56]
        future["rolling_mean_7"] = history[
            :, t - 34:t - 27
        ].mean(axis=1)
        future["rolling_mean_28"] = history[
            :, t - 55:t - 27
        ].mean(axis=1)

        future["weekday"] = int(cal["weekday"])
        future["month"] = int(cal["month"])
        future["year"] = int(cal["year"])
        future["is_event"] = int(cal["is_event"])
        future["wm_yr_wk"] = cal["wm_yr_wk"]

        future["snap"] = np.select(
            [
                future["state_id"].eq("CA"),
                future["state_id"].eq("TX"),
                future["state_id"].eq("WI")
            ],
            [
                cal["snap_CA"],
                cal["snap_TX"],
                cal["snap_WI"]
            ],
            default=0
        ).astype("int8")

        future = future.merge(
            prices[price_cols],
            on=["store_id", "item_id", "wm_yr_wk"],
            how="left"
        )

        pred = np.maximum(model.predict(future[features]), 0)

        outputs.append(pd.DataFrame({
            "id": future["id"].values,
            "d": d,
            "prediction": pred
        }))

    result = pd.concat(outputs, ignore_index=True)
    result_wide = result.pivot(
        index="id", columns="d", values="prediction"
    )
    result_wide = result_wide[future_days]
    return result_wide.to_numpy(dtype=np.float32)

# ============================================================
# 6. 正式训练：利用截至 d_1941 的训练目标
# ============================================================
del sampled, long_df, train_df, valid_df, data
gc.collect()

# 重新建立最终训练特征数据
# 使用已保留的 sales_eval 重新构造，避免依赖被删除的训练表
final_sample = sales_eval.iloc[sample_idx][
    meta_cols + [f"d_{i}" for i in range(TRAIN_START, 1942)]
].copy()

final_long = final_sample.melt(
    id_vars=meta_cols,
    var_name="d",
    value_name="sales"
)
final_long["day_num"] = final_long["d"].str[2:].astype("int16")

final_long = final_long.merge(
    calendar[calendar_features], on="d", how="left"
)
final_long = final_long.merge(
    prices[price_cols],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

final_long["snap"] = np.select(
    [
        final_long["state_id"].eq("CA"),
        final_long["state_id"].eq("TX"),
        final_long["state_id"].eq("WI")
    ],
    [
        final_long["snap_CA"],
        final_long["snap_TX"],
        final_long["snap_WI"]
    ],
    default=0
).astype("int8")

final_long = add_category_codes(final_long)
final_long = final_long.sort_values(
    ["id", "day_num"]
).reset_index(drop=True)

g = final_long.groupby("id", sort=False)["sales"]
final_long["lag_28"] = g.shift(28)
final_long["lag_56"] = g.shift(56)

final_long["rolling_mean_7"] = (
    final_long.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(7).mean())
)
final_long["rolling_mean_28"] = (
    final_long.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(28).mean())
)

final_data = final_long.dropna(
    subset=features + ["sales"]
)

final_model = LGBMRegressor(**params)
final_model.fit(
    final_data[features],
    final_data["sales"],
    categorical_feature=[]
)

print("\n最终模型训练完成。")

# ============================================================
# 7. 生成两组正式预测
# validation ID：用 d_1 至 d_1913 预测 d_1914 至 d_1941
# evaluation ID：用 d_1 至 d_1941 预测 d_1942 至 d_1969
# ============================================================
del final_sample, final_long, final_data
gc.collect()

val_history_cols = [
    c for c in sales_val.columns if c.startswith("d_")
]
eval_history_cols = [
    c for c in sales_eval.columns if c.startswith("d_")
]

val_future_days = [f"d_{i}" for i in range(1914, 1942)]
eval_future_days = [f"d_{i}" for i in range(1942, 1970)]

# 验证区间的正式预测必须使用未见过验证目标的模型
pred_val = forecast_direct(
    model_val, sales_val, val_history_cols, val_future_days
)

# 评价区间使用截至 d_1941 训练的最终模型
pred_eval = forecast_direct(
    final_model, sales_eval, eval_history_cols, eval_future_days
)

forecast_cols = [f"F{i}" for i in range(1, 29)]

val_result = pd.DataFrame(pred_val, columns=forecast_cols)
val_result.insert(0, "id", sales_val["id"].values)

eval_result = pd.DataFrame(pred_eval, columns=forecast_cols)
eval_result.insert(0, "id", sales_eval["id"].values)

all_predictions = pd.concat(
    [val_result, eval_result], ignore_index=True
)

submission = sample[["id"]].merge(
    all_predictions,
    on="id",
    how="left",
    validate="one_to_one"
)

assert submission.shape == sample.shape
assert submission["id"].equals(sample["id"])
assert np.isfinite(submission[forecast_cols].to_numpy()).all()

output_path = OUT + "submission_lgbm_features.csv"
submission.to_csv(output_path, index=False)

print("\n========== 提交文件检查 ==========")
print("文件：", output_path)
print("形状：", submission.shape)
print("预测缺失数：", submission[forecast_cols].isna().sum().sum())
display(submission.head())


In [ ]:

# ============================================================
# M5 A+B+C 一体化实验
# A: 增加历史销量滞后特征
# B: 诊断零销量商品
# C: 对比基线、旧 LightGBM、新 LightGBM
# ============================================================

import gc
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0. 检查前面成功运行的主程序变量
# ------------------------------------------------------------
required = ["sales_eval", "sales_val", "calendar", "prices",
            "sample_idx", "pred_val"]

missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(
        f"缺少变量：{missing}。请先重新运行完整的 LightGBM 主代码。"
    )

# 与原主程序保持一致
meta_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
price_cols = ["store_id", "item_id", "wm_yr_wk", "sell_price"]

if "params" not in globals():
    params = dict(
        objective="tweedie",
        tweedie_variance_power=1.2,
        n_estimators=350,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=80,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

# 即使 calendar 曾被原始 CSV 覆盖，也在这里重新准备日历特征
calendar = calendar.copy()
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday"] = calendar["date"].dt.dayofweek
calendar["month"] = calendar["date"].dt.month
calendar["year"] = calendar["date"].dt.year
calendar["is_event"] = (
    calendar["event_name_1"].notna() |
    calendar["event_name_2"].notna()
).astype("int8")

calendar_map = calendar.set_index("d")
calendar_features = [
    "d", "wm_yr_wk", "weekday", "month", "year",
    "is_event", "snap_CA", "snap_TX", "snap_WI"
]

# ------------------------------------------------------------
# 1. 固定相同的 5000 个商品和 28 天验证期
# ------------------------------------------------------------
sample_eval = sales_eval.iloc[sample_idx].reset_index(drop=True)

future_days = [f"d_{i}" for i in range(1914, 1942)]
history_days = [f"d_{i}" for i in range(1886, 1914)]

actual = sample_eval[future_days].to_numpy(dtype=np.float32)

# 季节性基线：过去 28 天直接对应未来 28 天
baseline_pred = sample_eval[history_days].to_numpy(dtype=np.float32)

# 旧模型预测由 forecast_direct 的 pivot 按 ID 排序返回。
# 因此按排序后的 validation ID 对齐，再转换为 evaluation ID。
old_ids_sorted = sorted(sales_val["id"].astype(str).tolist())

old_pred_df = pd.DataFrame(
    pred_val, columns=future_days
)
old_pred_df.insert(0, "id", old_ids_sorted)
old_pred_df["id"] = old_pred_df["id"].str.replace(
    "_validation", "_evaluation", regex=False
)

old_aligned = sample_eval[["id"]].merge(
    old_pred_df,
    on="id",
    how="left",
    validate="one_to_one"
)

old_pred = old_aligned[future_days].to_numpy(dtype=np.float32)

assert actual.shape == baseline_pred.shape == old_pred.shape
assert np.isfinite(old_pred).all(), "旧模型预测存在缺失值"

print("固定验证样本数：", len(sample_eval))
print("每个商品的预测天数：", len(future_days))
print("旧模型预测与真实销量已按 ID 对齐。")

# ------------------------------------------------------------
# A1. 构造训练数据，新增 lag_35、42、49、84
# 从 d_1438 开始，确保 d_1522 可以计算 lag_84
# ------------------------------------------------------------
print("\n========== A：构建新增特征 ==========")

start_day = 1438
end_day = 1941
day_cols = [f"d_{i}" for i in range(start_day, end_day + 1)]


long_new = sample_eval[meta_cols + day_cols].melt(
    id_vars=meta_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)


long_new["day_num"] = (
    long_new["d"].str[2:].astype("int16")
)

long_new = long_new.merge(
    calendar[calendar_features],
    on="d",
    how="left",
    validate="many_to_one"
)

long_new = long_new.merge(
    prices[price_cols],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

long_new["snap"] = np.select(
    [
        long_new["state_id"].eq("CA"),
        long_new["state_id"].eq("TX"),
        long_new["state_id"].eq("WI")
    ],
    [
        long_new["snap_CA"],
        long_new["snap_TX"],
        long_new["snap_WI"]
    ],
    default=0
).astype("int8")

long_new = long_new.sort_values(
    ["id", "day_num"]
).reset_index(drop=True)

g = long_new.groupby("id", sort=False)["sales"]

# 原有历史销量特征
long_new["lag_28"] = g.shift(28)
long_new["lag_56"] = g.shift(56)

# 新增特征
for lag in [35, 42, 49, 84]:
    long_new[f"lag_{lag}"] = g.shift(lag)

# 保持原模型的滚动特征定义
long_new["rolling_mean_7"] = (
    long_new.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(7).mean())
)

long_new["rolling_mean_28"] = (
    long_new.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(28).mean())
)

old_features = [
    "weekday", "month", "year", "is_event", "snap",
    "sell_price", "lag_28", "lag_56",
    "rolling_mean_7", "rolling_mean_28"
]

new_features = old_features + [
    "lag_35", "lag_42", "lag_49", "lag_84"
]

# 训练目标严格限制在 d_1522 至 d_1913
train_new = long_new[
    (long_new["day_num"] >= 1522) &
    (long_new["day_num"] < 1914)
].dropna(subset=new_features + ["sales"]).copy()

print("新增模型训练行数：", len(train_new))
print("旧特征数：", len(old_features))
print("新特征数：", len(new_features))
print("新增特征：", ["lag_35", "lag_42", "lag_49", "lag_84"])

# ------------------------------------------------------------
# A2. 训练新增特征模型
# ------------------------------------------------------------
print("\n开始训练新 LightGBM，请等待完成……")

model_new = LGBMRegressor(**params)
model_new.fit(
    train_new[new_features],
    train_new["sales"],
    categorical_feature=[]
)

print("新模型训练完成。")

# ------------------------------------------------------------
# A3. 使用历史真实销量直接预测验证期 28 天
# 每一天的特征只使用该目标日之前至少 28 天的真实销量
# ------------------------------------------------------------
def forecast_direct_extended(model, sales_df, future_days, features):
    hist_cols = [
        f"d_{i}" for i in range(1, 1942)
    ]
    hist = sales_df[hist_cols].to_numpy(dtype=np.float32)
    meta_future = sales_df[meta_cols].copy()

    outputs = []

    for d in future_days:
        cal = calendar_map.loc[d]
        t = int(d[2:]) - 1

        future = meta_future.copy()
        future["d"] = d
        future["day_num"] = int(d[2:])

        # 与训练时一致的滞后特征
        for lag in [28, 35, 42, 49, 56, 84]:
            future[f"lag_{lag}"] = hist[:, t - lag]

        future["rolling_mean_7"] = hist[:, t-34:t-27].mean(axis=1)
        future["rolling_mean_28"] = hist[:, t-55:t-27].mean(axis=1)

        future["weekday"] = int(cal["weekday"])
        future["month"] = int(cal["month"])
        future["year"] = int(cal["year"])
        future["is_event"] = int(cal["is_event"])
        future["wm_yr_wk"] = cal["wm_yr_wk"]

        future["snap"] = np.select(
            [
                future["state_id"].eq("CA"),
                future["state_id"].eq("TX"),
                future["state_id"].eq("WI")
            ],
            [
                cal["snap_CA"],
                cal["snap_TX"],
                cal["snap_WI"]
            ],
            default=0
        ).astype("int8")

        future = future.merge(
            prices[price_cols],
            on=["store_id", "item_id", "wm_yr_wk"],
            how="left",
            validate="many_to_one"
        )

        pred = np.maximum(
            model.predict(future[features]), 0
        )
        outputs.append(pred)

    # shape: 商品数 × 28 天，行顺序与 sales_df 一致
    return np.column_stack(outputs).astype(np.float32)


new_pred = forecast_direct_extended(
    model_new, sample_eval, future_days, new_features
)

assert new_pred.shape == actual.shape
assert np.isfinite(new_pred).all()

# ------------------------------------------------------------
# C. 同样本、同验证期，对比三个模型
# ------------------------------------------------------------
print("\n========== C：模型公平比较 ==========")

def score_model(name, y, p):
    mae = mean_absolute_error(y.ravel(), p.ravel())
    rmse = np.sqrt(mean_squared_error(y.ravel(), p.ravel()))
    return {"模型": name, "MAE": mae, "RMSE": rmse}

results = pd.DataFrame([
    score_model("季节性基线", actual, baseline_pred),
    score_model("旧 LightGBM", actual, old_pred),
    score_model("新 LightGBM（增加滞后特征）", actual, new_pred)
])

display(results.round(4))

old_mae = results.loc[results["模型"] == "旧 LightGBM", "MAE"].iloc[0]
new_mae = results.loc[results["模型"] == "新 LightGBM（增加滞后特征）", "MAE"].iloc[0]
old_rmse = results.loc[results["模型"] == "旧 LightGBM", "RMSE"].iloc[0]
new_rmse = results.loc[results["模型"] == "新 LightGBM（增加滞后特征）", "RMSE"].iloc[0]

print(f"新模型 MAE 相对旧模型变化：{(new_mae / old_mae - 1) * 100:+.2f}%")
print(f"新模型 RMSE 相对旧模型变化：{(new_rmse / old_rmse - 1) * 100:+.2f}%")

# ------------------------------------------------------------
# B. 零销量商品诊断
# 根据验证开始前的最近 28 天销量分组
# ------------------------------------------------------------
print("\n========== B：零销量商品诊断 ==========")

recent_28 = sample_eval[history_days].to_numpy(dtype=np.float32)
zero_rate = (recent_28 == 0).mean(axis=1)

segments = {
    "近28天全零": zero_rate == 1.0,
    "近28天至少80%为零": (zero_rate >= 0.8) & (zero_rate < 1.0),
    "其他商品": zero_rate < 0.8
}

segment_rows = []

for name, mask in segments.items():
    count = int(mask.sum())

    if count == 0:
        continue

    y = actual[mask]
    base = baseline_pred[mask]
    old = old_pred[mask]
    new = new_pred[mask]

    segment_rows.append({
        "商品分组": name,
        "商品数": count,
        "真实平均日销量": float(y.mean()),
        "基线平均预测": float(base.mean()),
        "旧模型平均预测": float(old.mean()),
        "新模型平均预测": float(new.mean()),
        "旧模型MAE": float(mean_absolute_error(y.ravel(), old.ravel())),
        "新模型MAE": float(mean_absolute_error(y.ravel(), new.ravel()))
    })

segment_results = pd.DataFrame(segment_rows)
display(segment_results.round(4))

# ------------------------------------------------------------
# 新特征重要性
# ------------------------------------------------------------
print("\n========== 新模型特征重要性 ==========")

importance = pd.DataFrame({
    "特征": new_features,
    "重要性": model_new.feature_importances_
}).sort_values("重要性", ascending=False)

display(importance.reset_index(drop=True))

# 清理占用内存的大型训练表
del long_new, train_new
gc.collect()

print("\nA+B+C 实验全部完成。")
print("原有 submission_lgbm_features.csv 未被覆盖。")


In [ ]:

import gc
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# 1. 实验设置：与上一轮保持一致
# ============================================================
BASE_FEATURES = [
    "weekday", "month", "year", "is_event", "snap", "sell_price",
    "lag_28", "lag_56", "rolling_mean_7", "rolling_mean_28",
    "lag_35", "lag_42", "lag_49", "lag_84"
]

NEW_FEATURES = ["lag_35", "lag_42", "lag_49", "lag_84"]

META_COLS = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

TRAIN_START = 1522
TRAIN_END = 1913
VALID_START = 1914
VALID_END = 1941

print("========== 1. 重建实验数据 ==========")

# 只需要覆盖最长滞后所需的历史数据，以及训练和验证期
DAY_START = TRAIN_START - 84
DAY_END = VALID_END
DAY_COLS = [f"d_{i}" for i in range(DAY_START, DAY_END + 1)]

missing_cols = [c for c in META_COLS + DAY_COLS
                if c not in sample_eval.columns]
if missing_cols:
    raise ValueError(f"sample_eval 缺少必要列：{missing_cols[:10]}")

long_exp = sample_eval[META_COLS + DAY_COLS].melt(
    id_vars=META_COLS,
    value_vars=DAY_COLS,
    var_name="d",
    value_name="sales"
)

long_exp["day_num"] = long_exp["d"].str[2:].astype("int16")
long_exp = long_exp.sort_values(
    ["id", "day_num"], kind="mergesort"
).reset_index(drop=True)

# ============================================================
# 2. 日历特征与价格
# ============================================================
cal = calendar.copy()
if "d" not in cal.columns:
    cal = cal.reset_index()

if "month" not in cal.columns:
    cal["month"] = pd.to_datetime(cal["date"]).dt.month
if "year" not in cal.columns:
    cal["year"] = pd.to_datetime(cal["date"]).dt.year
if "weekday" not in cal.columns:
    cal["weekday"] = pd.to_datetime(cal["date"]).dt.dayofweek
if "is_event" not in cal.columns:
    event_cols = [
        c for c in ["event_name_1", "event_name_2"]
        if c in cal.columns
    ]
    if event_cols:
        cal["is_event"] = cal[event_cols].notna().any(axis=1).astype("int8")
    else:
        cal["is_event"] = 0

calendar_cols = [
    "d", "wm_yr_wk", "weekday", "month", "year", "is_event"
]
snap_cols = [c for c in ["snap_CA", "snap_TX", "snap_WI"]
             if c in cal.columns]
calendar_cols += snap_cols

cal_small = cal[calendar_cols].drop_duplicates("d")
long_exp = long_exp.merge(
    cal_small, on="d", how="left", validate="many_to_one"
)

# 按商品所属州选择对应 SNAP 指标
snap_map = {
    "CA": "snap_CA",
    "TX": "snap_TX",
    "WI": "snap_WI"
}
long_exp["snap"] = 0
for state, snap_col in snap_map.items():
    if snap_col in long_exp.columns:
        mask = long_exp["state_id"].eq(state)
        long_exp.loc[mask, "snap"] = long_exp.loc[mask, snap_col]

# 加入价格
price_small = prices[
    ["store_id", "item_id", "wm_yr_wk", "sell_price"]
].drop_duplicates(["store_id", "item_id", "wm_yr_wk"])

long_exp = long_exp.merge(
    price_small,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

# ============================================================
# 3. 构建滞后特征与滚动均值
# ============================================================
print("构建滞后特征与滚动均值……")

grp = long_exp.groupby("id", sort=False)["sales"]

for lag in [28, 35, 42, 49, 56, 84]:
    long_exp[f"lag_{lag}"] = grp.shift(lag)

# 与直接预测 28 天的设定一致：滚动窗口只使用目标日前至少 28 天的数据
shifted_28 = long_exp.groupby("id", sort=False)["sales"].shift(28)

long_exp["rolling_mean_7"] = (
    shifted_28.groupby(long_exp["id"], sort=False)
    .rolling(7, min_periods=7).mean()
    .reset_index(level=0, drop=True)
)
long_exp["rolling_mean_28"] = (
    shifted_28.groupby(long_exp["id"], sort=False)
    .rolling(28, min_periods=28).mean()
    .reset_index(level=0, drop=True)
)

# 训练期：d_1522 至 d_1913
train_exp = long_exp[
    long_exp["day_num"].between(TRAIN_START, TRAIN_END)
].copy()

# 验证期：d_1914 至 d_1941
valid_exp = long_exp[
    long_exp["day_num"].between(VALID_START, VALID_END)
].copy()

# 训练集特征缺失的行统一剔除，保证各组使用相同训练样本
train_exp = train_exp.dropna(subset=BASE_FEATURES + ["sales"]).copy()

# 验证集保持相同的商品和日期；若有缺失，明确报错而不静默删样本
if valid_exp[BASE_FEATURES].isna().any().any():
    bad = valid_exp[BASE_FEATURES].isna().sum()
    raise ValueError(
        "验证特征存在缺失，请检查日历、价格或滞后特征：\n"
        + str(bad[bad > 0])
    )

X_valid = valid_exp[BASE_FEATURES]
y_valid = valid_exp["sales"].to_numpy()

X_train_all = train_exp[BASE_FEATURES]
y_train = train_exp["sales"].to_numpy()

print("训练行数：", len(train_exp))
print("验证行数：", len(valid_exp))
print("验证商品数：", valid_exp["id"].nunique())
print("验证天数：", valid_exp["day_num"].nunique())

# ============================================================
# 4. 对照模型 + 四组消融模型
# ============================================================
print("\n========== 2. 开始消融实验 ==========")

experiments = [
    ("14特征完整模型", None),
    ("删除 lag_35", "lag_35"),
    ("删除 lag_42", "lag_42"),
    ("删除 lag_49", "lag_49"),
    ("删除 lag_84", "lag_84")
]

results = []
ablation_models = {}

for exp_name, drop_feature in experiments:
    feats = BASE_FEATURES.copy()
    if drop_feature is not None:
        feats.remove(drop_feature)

    print(f"\n开始训练：{exp_name}；特征数：{len(feats)}")

    model = lgb.LGBMRegressor(**params)
    model.fit(
        train_exp[feats],
        y_train
    )

    pred = model.predict(valid_exp[feats])
    mae = mean_absolute_error(y_valid, pred)
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    results.append({
        "实验": exp_name,
        "特征数": len(feats),
        "MAE": mae,
        "RMSE": rmse
    })
    ablation_models[exp_name] = model

    print(f"{exp_name} 完成：MAE={mae:.5f}，RMSE={rmse:.5f}")

    del model, pred
    gc.collect()

# ============================================================
# 5. 汇总与解释
# ============================================================
result_df = pd.DataFrame(results)
full_mae = result_df.loc[
    result_df["实验"] == "14特征完整模型", "MAE"
].iloc[0]
full_rmse = result_df.loc[
    result_df["实验"] == "14特征完整模型", "RMSE"
].iloc[0]

result_df["相对完整模型 MAE变化(%)"] = (
    (result_df["MAE"] / full_mae - 1) * 100
)
result_df["相对完整模型 RMSE变化(%)"] = (
    (result_df["RMSE"] / full_rmse - 1) * 100
)

print("\n========== 消融实验最终结果 ==========")
display(result_df.round(5))

print("\n解释：")
print("删除某特征后，MAE/RMSE 下降（变化为负）意味着该消融模型在本次验证集上更好。")
print("删除某特征后，MAE/RMSE 上升（变化为正）意味着该特征可能提供了有效信息。")

# 保存实验结果为新文件，不覆盖原提交文件
result_df.to_csv(
    "/kaggle/working/feature_ablation_results.csv",
    index=False
)
print("\n实验结果已保存至 /kaggle/working/feature_ablation_results.csv")

# 释放较大的中间表
del long_exp, train_exp, valid_exp, X_train_all
gc.collect()


In [ ]:

# 第一步：核对现有模型与特征信息
import numpy as np
import pandas as pd

for name in ["model_new", "model_val", "final_model"]:
    model = globals().get(name)
    print(f"\n{'=' * 55}\n模型：{name}")

    if model is None:
        print("不存在")
        continue

    # 兼容 LightGBM sklearn 接口
    booster = model.booster_ if hasattr(model, "booster_") else model

    if hasattr(booster, "feature_name"):
        feature_names = booster.feature_name()
    elif hasattr(model, "feature_names_in_"):
        feature_names = list(model.feature_names_in_)
    else:
        feature_names = []

    print("特征数量：", len(feature_names))
    print("特征列表：", feature_names)

    if hasattr(booster, "feature_importance"):
        for importance_type in ["gain", "split"]:
            importance = booster.feature_importance(
                importance_type=importance_type
            )
            print(f"{importance_type} 前5项：")
            print(
                pd.DataFrame({
                    "feature": feature_names,
                    importance_type: importance
                })
                .sort_values(importance_type, ascending=False)
                .head(5)
                .to_string(index=False)
            )

print("\n预测与标签形状：")
for name in ["actual", "baseline_pred", "pred_val", "pred_eval"]:
    obj = globals().get(name)
    if obj is not None:
        print(name, np.shape(obj))


In [ ]:

# 第二步：寻找现有验证数据，避免重复训练

import pandas as pd
import numpy as np

print("=== 当前内存中的 DataFrame ===")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)
        has_target = "sales" in cols
        feature_hits = [
            c for c in [
                "weekday", "month", "sell_price",
                "lag_28", "lag_84", "rolling_mean_28"
            ] if c in cols
        ]
        if has_target or len(feature_hits) >= 3:
            print(
                f"{name}: shape={obj.shape}, "
                f"含sales列={has_target}, "
                f"特征匹配={feature_hits}"
            )

print("\n=== 当前内存中的二维数组 ===")
for name, obj in list(globals().items()):
    if isinstance(obj, (np.ndarray, pd.DataFrame)):
        shape = getattr(obj, "shape", None)
        if shape is not None and len(shape) == 2:
            if shape[0] in [5000, 30490] or shape[1] in [10, 11, 14, 28]:
                print(f"{name}: shape={shape}")


In [ ]:

# 第三步：核验验证集、模型预测和真实销量的对应关系

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 核对模型特征顺序
model_features = list(model_new.booster_.feature_name())
valid_features = list(X_valid.columns)

print("X_valid 形状：", X_valid.shape)
print("模型特征数：", len(model_features))
print("X_valid 特征数：", len(valid_features))
print("特征顺序完全一致：", model_features == valid_features)

if model_features != valid_features:
    print("\n模型特征：", model_features)
    print("验证集特征：", valid_features)

# 2. 用现有模型预测 X_valid
pred_from_X = model_new.predict(X_valid)

print("\nX_valid 预测形状：", np.shape(pred_from_X))
print("new_pred 形状：", np.shape(new_pred))
print("actual 形状：", np.shape(actual))

# 3. 只有形状符合时才比较指标
if (
    np.size(pred_from_X) == actual.size
    and np.size(new_pred) == actual.size
):
    y_true = np.asarray(actual).reshape(-1)
    pred_X = np.asarray(pred_from_X).reshape(-1)
    pred_saved = np.asarray(new_pred).reshape(-1)

    def metrics(y_true, pred):
        return {
            "MAE": mean_absolute_error(y_true, pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, pred))
        }

    print("\n=== 指标对照（用于核验对齐情况）===")
    print("model_new 直接预测 X_valid：", metrics(y_true, pred_X))
    print("已有 new_pred：", metrics(y_true, pred_saved))

    print(
        "\n两个预测序列平均绝对差：",
        np.mean(np.abs(pred_X - pred_saved))
    )
else:
    print("\n形状不匹配：暂不比较指标，需要进一步核对数据排列。")


In [ ]:

# 第四步：核对 X_valid 与已有验证结果的对应关系

print("=== X_valid 前5行 ===")
display(X_valid.head())

print("\n=== 当前可能的目标变量 ===")
for name in ["valid_df", "train_df", "data", "val_result", "eval_result"]:
    obj = globals().get(name)
    if obj is not None:
        print(f"\n{name}: type={type(obj).__name__}, shape={getattr(obj, 'shape', None)}")
        if hasattr(obj, "columns"):
            print("列名：", list(obj.columns)[:35])
            if "sales" in obj.columns:
                print("sales 前5个值：", obj["sales"].head().tolist())

print("\n=== 验证预测矩阵 ===")
print("actual:", np.shape(actual))
print("new_pred:", np.shape(new_pred))
print("old_pred:", np.shape(old_pred))

print("\n=== 14特征模型训练信息 ===")
print("模型类型：", type(model_new).__name__)
print("训练特征数：", len(model_new.booster_.feature_name()))
print("训练参数：", model_new.get_params())


In [ ]:

# 第五步：查找验证集的标签和构造来源
import pandas as pd
import numpy as np

print("=== 可能的验证标签 ===")
for name, obj in list(globals().items()):
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        cols = list(obj.columns) if isinstance(obj, pd.DataFrame) else []
        if (
            name.lower().startswith(("y_", "valid", "val", "target"))
            or "sales" in cols
        ):
            print(
                name,
                "type =", type(obj).__name__,
                "shape =", getattr(obj, "shape", None),
                "columns =", cols[:20]
            )

print("\n=== X_valid 索引信息 ===")
print("索引类型：", type(X_valid.index).__name__)
print("索引是否唯一：", X_valid.index.is_unique)
print("索引前10个：", X_valid.index[:10].tolist())
print("索引最后10个：", X_valid.index[-10:].tolist())

print("\n=== 当前验证相关的标量/数组变量 ===")
for name in [
    "FIT_START", "VALID_START", "VALID_END",
    "features", "features_new", "sample_idx",
    "y_valid", "valid_df", "data"
]:
    if name in globals():
        obj = globals()[name]
        print(
            name,
            "=",
            repr(obj)[:500] if not hasattr(obj, "shape") else
            f"type={type(obj).__name__}, shape={obj.shape}"
        )
    else:
        print(name, "= 不存在")


In [ ]:

# 第六步：14特征模型的 gain / split / permutation importance
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 确保特征顺序与真实标签对应
X_imp = X_valid[model_new.booster_.feature_name()]
y_imp = np.asarray(y_valid).reshape(-1)

assert len(X_imp) == len(y_imp), (
    f"行数不一致：X={len(X_imp)}, y={len(y_imp)}"
)

# 2. 固定抽样，降低计算耗时
rng = np.random.RandomState(42)
n = min(30000, len(X_imp))
idx = rng.choice(len(X_imp), size=n, replace=False)

X_sample = X_imp.iloc[idx].copy()
y_sample = y_imp[idx]

# 3. 基准验证表现
pred_sample = model_new.predict(X_sample)

mae_base = mean_absolute_error(y_sample, pred_sample)
rmse_base = np.sqrt(mean_squared_error(y_sample, pred_sample))

print("验证抽样行数：", n)
print(f"基准 MAE： {mae_base:.6f}")
print(f"基准 RMSE：{rmse_base:.6f}")

# 4. 读取 gain 和 split
booster = model_new.booster_
gain = booster.feature_importance(importance_type="gain")
split = booster.feature_importance(importance_type="split")

importance_df = pd.DataFrame({
    "feature": booster.feature_name(),
    "gain": gain,
    "split": split
})

# 5. 计算置换重要性
# scoring='neg_mean_absolute_error'
# importance_mean > 0 表示打乱该特征后，MAE平均变差
perm = permutation_importance(
    model_new,
    X_sample,
    y_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

importance_df["perm_mae_increase"] = perm.importances_mean
importance_df["perm_std"] = perm.importances_std

# 6. 方便阅读：归一化 gain，另外按置换重要性排序
gain_total = importance_df["gain"].sum()
importance_df["gain_pct"] = (
    importance_df["gain"] / gain_total * 100
    if gain_total > 0 else 0
)

importance_df = importance_df.sort_values(
    "perm_mae_increase", ascending=False
).reset_index(drop=True)

print("\n=== 特征重要性综合表 ===")
display(
    importance_df.round({
        "gain": 2,
        "gain_pct": 2,
        "perm_mae_increase": 6,
        "perm_std": 6
    })
)

# 7. 保存独立结果，不覆盖原提交文件
output_path = "/kaggle/working/importance_14_features.csv"
importance_df.to_csv(output_path, index=False)
print("\n结果已保存：", output_path)


In [ ]:

# 第七步：检查是否保留了训练集特征，暂不训练

import pandas as pd
import numpy as np

for name, obj in list(globals().items()):
    if name.startswith("__"):
        continue

    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)
        feature_count = sum(
            c in cols for c in [
                "weekday", "month", "year", "sell_price",
                "lag_28", "lag_56", "lag_84",
                "rolling_mean_7", "rolling_mean_28"
            ]
        )
        if feature_count >= 5:
            print(
                f"{name}: shape={obj.shape}, "
                f"特征匹配数={feature_count}, "
                f"含sales列={'sales' in cols}"
            )

    elif isinstance(obj, (np.ndarray, pd.Series)):
        low = name.lower()
        if any(k in low for k in ["train", "valid", "target", "label", "y_"]):
            print(
                f"{name}: type={type(obj).__name__}, "
                f"shape={getattr(obj, 'shape', None)}"
            )

print("\n=== 11特征候选清单 ===")
features_11 = [
    "weekday", "month", "year", "is_event", "snap",
    "sell_price", "lag_28", "lag_56",
    "rolling_mean_7", "rolling_mean_28", "lag_84"
]
print(features_11)
print("特征数量：", len(features_11))


In [ ]:

# 第八步：重建训练特征，训练11特征候选模型并公平比较
import gc
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------- 1. 定义固定的特征集合 ----------
features_10 = [
    "weekday", "month", "year", "is_event", "snap",
    "sell_price", "lag_28", "lag_56",
    "rolling_mean_7", "rolling_mean_28"
]

features_11 = features_10 + ["lag_84"]

assert len(features_11) == 11
assert "lag_84" in X_valid.columns
assert len(X_valid) == len(y_valid)

# ---------- 2. 只重建训练期需要的数据 ----------
meta_cols_local = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

train_day_cols = [f"d_{i}" for i in range(1466, 1914)]

sampled_train = sales_eval.iloc[sample_idx][
    meta_cols_local + train_day_cols
].copy()

train_long_11 = sampled_train.melt(
    id_vars=meta_cols_local,
    value_vars=train_day_cols,
    var_name="d",
    value_name="sales"
)

train_long_11["day_num"] = (
    train_long_11["d"].str[2:].astype("int16")
)

calendar_cols_local = [
    "d", "wm_yr_wk", "weekday", "month", "year",
    "is_event", "snap_CA", "snap_TX", "snap_WI"
]

train_long_11 = train_long_11.merge(
    calendar[calendar_cols_local],
    on="d",
    how="left",
    validate="many_to_one"
)

price_cols_local = [
    "store_id", "item_id", "wm_yr_wk", "sell_price"
]

train_long_11 = train_long_11.merge(
    prices[price_cols_local],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

# SNAP 按商品所在州匹配
train_long_11["snap"] = np.select(
    [
        train_long_11["state_id"].eq("CA"),
        train_long_11["state_id"].eq("TX"),
        train_long_11["state_id"].eq("WI")
    ],
    [
        train_long_11["snap_CA"],
        train_long_11["snap_TX"],
        train_long_11["snap_WI"]
    ],
    default=0
).astype("int8")

train_long_11 = train_long_11.sort_values(
    ["id", "day_num"]
).reset_index(drop=True)

# ---------- 3. 构造历史特征 ----------
sales_group = train_long_11.groupby("id", sort=False)["sales"]

train_long_11["lag_28"] = sales_group.shift(28)
train_long_11["lag_56"] = sales_group.shift(56)
train_long_11["lag_84"] = sales_group.shift(84)

train_long_11["rolling_mean_7"] = (
    train_long_11.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(7).mean())
)

train_long_11["rolling_mean_28"] = (
    train_long_11.groupby("id", sort=False)["sales"]
    .transform(lambda s: s.shift(28).rolling(28).mean())
)

# 与原实验一致：训练目标限定在 d_1522 到 d_1913
# lag_84 缺失的早期行会被删除，因此训练样本数会减少
train_mask_11 = (
    (train_long_11["day_num"] >= FIT_START) &
    (train_long_11["day_num"] < VALID_START)
)

train_data_11 = train_long_11.loc[
    train_mask_11
].dropna(subset=features_11 + ["sales"])

X_train_11 = train_data_11[features_11]
y_train_11 = train_data_11["sales"]

print("训练特征矩阵：", X_train_11.shape)
print("训练标签：", y_train_11.shape)
print("11特征候选：", features_11)

# ---------- 4. 训练11特征模型 ----------
# 复制现有参数，避免改变旧模型的参数字典
params_11 = model_val.get_params().copy()

model_11 = LGBMRegressor(**params_11)
model_11.fit(X_train_11, y_train_11)

print("\n11特征模型训练完成。")

# ---------- 5. 在完全相同的验证行上比较 ----------
X_valid_10 = X_valid[features_10]
X_valid_11 = X_valid[features_11]
y_valid_array = np.asarray(y_valid).reshape(-1)

pred_10_same = np.maximum(model_val.predict(X_valid_10), 0)
pred_11_same = np.maximum(model_11.predict(X_valid_11), 0)

def evaluate_predictions(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred))
    }

results_same = pd.DataFrame([
    {
        "model": "旧10特征",
        **evaluate_predictions(y_valid_array, pred_10_same)
    },
    {
        "model": "新11特征（增加lag_84）",
        **evaluate_predictions(y_valid_array, pred_11_same)
    }
])

print("\n=== 同一验证集上的公平比较 ===")
display(results_same.round(6))

mae_change = (
    results_same.loc[1, "MAE"] - results_same.loc[0, "MAE"]
)
rmse_change = (
    results_same.loc[1, "RMSE"] - results_same.loc[0, "RMSE"]
)

print(f"MAE变化（新模型 - 旧模型）：{mae_change:+.6f}")
print(f"RMSE变化（新模型 - 旧模型）：{rmse_change:+.6f}")

# 保存结果，不覆盖原提交文件
results_same.to_csv(
    "/kaggle/working/comparison_10_vs_11_features.csv",
    index=False
)

# 释放大型临时表，保留 model_11 和 X_train_11 供后续分析
del sampled_train, train_long_11, train_data_11
gc.collect()

print("\n比较结果已保存：/kaggle/working/comparison_10_vs_11_features.csv")


In [ ]:

# 第九步：按需求类型比较10特征与11特征模型
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. 按与 X_valid 一致的商品ID顺序整理历史销量
# X_valid 来自按 id、day_num 排序后的验证长表
history_cols = [f"d_{i}" for i in range(1886, 1914)]

sample_history = sales_eval.iloc[sample_idx][
    ["id"] + history_cols
].sort_values("id").reset_index(drop=True)

assert len(sample_history) == 5000
assert len(y_valid_array) == 5000 * 28
assert len(pred_10_same) == 5000 * 28
assert len(pred_11_same) == 5000 * 28

recent_history = sample_history[history_cols].to_numpy()
zero_rate = (recent_history == 0).mean(axis=1)

# 2. 划分商品类型
group = np.where(
    zero_rate == 1,
    "近28天全零",
    np.where(zero_rate >= 0.8, "间歇性需求", "其他商品")
)

# 3. 恢复商品 × 28天矩阵
y_mat = y_valid_array.reshape(5000, 28)
pred10_mat = pred_10_same.reshape(5000, 28)
pred11_mat = pred_11_same.reshape(5000, 28)

# 4. 每个分组分别计算指标
rows = []

for group_name in ["近28天全零", "间歇性需求", "其他商品"]:
    mask = group == group_name
    y_g = y_mat[mask].reshape(-1)
    p10_g = pred10_mat[mask].reshape(-1)
    p11_g = pred11_mat[mask].reshape(-1)

    rows.append({
        "商品组": group_name,
        "商品数": int(mask.sum()),
        "真实平均日销量": float(y_g.mean()),
        "10特征_MAE": mean_absolute_error(y_g, p10_g),
        "11特征_MAE": mean_absolute_error(y_g, p11_g),
        "MAE变化": mean_absolute_error(y_g, p11_g)
                   - mean_absolute_error(y_g, p10_g),
        "10特征_RMSE": np.sqrt(mean_squared_error(y_g, p10_g)),
        "11特征_RMSE": np.sqrt(mean_squared_error(y_g, p11_g)),
    })

group_results = pd.DataFrame(rows)

print("=== 各需求类型的验证表现 ===")
display(group_results.round(5))

print("\n说明：MAE变化、RMSE若下降为负值，表示11特征模型改善；")
print("若上升为正值，表示11特征模型在该组变差。")

group_results.to_csv(
    "/kaggle/working/group_metrics_10_vs_11.csv",
    index=False
)
print("\n结果已保存：/kaggle/working/group_metrics_10_vs_11.csv")


In [ ]:
import pandas as pd
import numpy as np

importance_11 = pd.DataFrame({
    "feature": model_11.booster_.feature_name(),
    "gain": model_11.booster_.feature_importance(importance_type="gain"),
    "split": model_11.booster_.feature_importance(importance_type="split")
})

importance_11["gain_pct"] = (
    importance_11["gain"] / importance_11["gain"].sum() * 100
)

importance_11 = importance_11.sort_values(
    "gain_pct", ascending=False
).reset_index(drop=True)

display(importance_11.round(4))

print("lag_84 在 gain 排名中的位置：",
      importance_11.index[importance_11["feature"] == "lag_84"].tolist()[0] + 1)

print("lag_84 的 gain 占比：",
      round(importance_11.loc[importance_11["feature"] == "lag_84",
                              "gain_pct"].iloc[0], 3), "%")

In [ ]:
for name in [
    "train_long_11",
    "sample_eval",
    "sample_idx",
    "model_11",
    "X_valid",
    "y_valid"
]:
    obj = globals().get(name)
    print(
        f"{name}:",
        "不存在" if obj is None else
        f"shape={obj.shape}" if hasattr(obj, "shape") else type(obj).__name__
    )

if "train_long_11" in globals():
    print("\ntrain_long_11 列名：")
    print(train_long_11.columns.tolist())

In [ ]:

import numpy as np
import pandas as pd

# ============================================================
# 第二阶段：长期销售水平特征
# 使用验证开始前的数据，避免泄漏未来销量
# ============================================================

assert "sample_eval" in globals(), "缺少 sample_eval"
assert "sample_idx" in globals(), "缺少 sample_idx"
assert "model_11" in globals(), "缺少 model_11"

# M5 验证集：d_1914 至 d_1941，共 28 天
VALID_START = 1914
VALID_END = 1941
HORIZON = 28

# 取抽样商品的历史销量
day_cols = [
    f"d_{i}" for i in range(1, VALID_START)
]

missing_cols = [
    c for c in day_cols
    if c not in sample_eval.columns
]
assert not missing_cols, f"缺少历史日期列：{missing_cols[:5]}"

history = sample_eval[day_cols].to_numpy(dtype=np.float32)

print("抽样商品数：", len(sample_eval))
print("历史销量矩阵：", history.shape)
print("历史截止日期：d_1913")
print("验证日期：d_1914 至 d_1941")
print("11特征模型特征数：",
      len(model_11.booster_.feature_name()))

# 三个长期销售水平候选特征
# 都只使用 d_1913 及以前的历史数据
level_56 = history[:, -56:].mean(axis=1)

zero_ratio_56 = (
    history[:, -56:] == 0
).mean(axis=1)

nonzero_mean_56 = np.divide(
    (history[:, -56:] * (history[:, -56:] > 0)).sum(axis=1),
    (history[:, -56:] > 0).sum(axis=1),
    out=np.zeros(len(history), dtype=np.float32),
    where=(history[:, -56:] > 0).sum(axis=1) > 0
)

long_features = pd.DataFrame({
    "id": sample_eval["id"].to_numpy(),
    "mean_sales_56": level_56,
    "zero_ratio_56": zero_ratio_56,
    "nonzero_mean_56": nonzero_mean_56
})

print("\n========== 新特征检查 ==========")
display(long_features.head(10))

print("\n近56天全零商品数：",
      int((zero_ratio_56 == 1).sum()))
print("近56天零销量比例中位数：",
      round(float(np.median(zero_ratio_56)), 4))
print("近56天平均销量中位数：",
      round(float(np.median(level_56)), 4))

print("\n特征构造完成，尚未训练新模型。")


In [ ]:
import numpy as np
import pandas as pd

check_names = [
    "sample_eval", "sample_idx", "sales_eval", "sales_val",
    "model_11", "features_11", "X_train_11", "y_train_11",
    "X_valid", "y_valid", "train_long_11",
    "actual", "params"
]

for name in check_names:
    obj = globals().get(name)

    if obj is None:
        print(f"{name}: 不存在")
    elif hasattr(obj, "shape"):
        print(f"{name}: shape={obj.shape}")
        if isinstance(obj, pd.DataFrame):
            print("  columns:", obj.columns.tolist()[:20])
    elif isinstance(obj, (list, tuple, np.ndarray)):
        print(f"{name}: 类型={type(obj).__name__}, 长度={len(obj)}")
        if name == "features_11":
            print("  特征:", obj)
    else:
        print(f"{name}: 类型={type(obj).__name__}")

print("\n--- 日期范围与商品数量检查 ---")

if "sample_eval" in globals():
    df = sample_eval
    day_cols = [
        c for c in df.columns
        if isinstance(c, str)
        and c.startswith("d_")
        and c[2:].isdigit()
    ]
    day_nums = sorted(int(c[2:]) for c in day_cols)

    print("商品行数:", len(df))
    print("日期列数量:", len(day_nums))
    print("最早日期:", min(day_nums) if day_nums else None)
    print("最晚日期:", max(day_nums) if day_nums else None)
    print(
        "商品 ID 是否唯一:",
        df["id"].is_unique if "id" in df.columns else "无 id 列"
    )

for name in ["X_train_11", "y_train_11", "X_valid", "y_valid"]:
    obj = globals().get(name)
    if obj is not None and hasattr(obj, "__len__"):
        print(f"{name} 行数:", len(obj))

In [ ]:
import numpy as np

assert old_pred.shape == actual.shape == (5000, 28)
assert new_pred.shape == actual.shape == (5000, 28)
assert np.isfinite(old_pred).all()
assert np.isfinite(new_pred).all()
assert np.isfinite(actual).all()

print("基准预测与真实销量形状一致：", actual.shape)
print("旧模型 MAE：", np.mean(np.abs(old_pred - actual)))
print("旧模型 RMSE：", np.sqrt(np.mean((old_pred - actual) ** 2)))
print("新模型 MAE：", np.mean(np.abs(new_pred - actual)))
print("新模型 RMSE：", np.sqrt(np.mean((new_pred - actual) ** 2)))
print("历史销量列数：", sum(
    c.startswith("d_") and c[2:].isdigit()
    for c in sample_eval.columns
))
print("训练截止日期：d_1913；验证期：d_1914–d_1941")

In [ ]:
import numpy as np
import pandas as pd

# 只使用预测起点之前的销量
TRAIN_END = 1913
HORIZON = 28
WINDOW = 56

day_cols = [f"d_{i}" for i in range(1, TRAIN_END + 1)]
sales_hist = sample_eval[day_cols].to_numpy(dtype=np.float32)

# 每个商品最近 56 天的历史销量
recent_56 = sales_hist[:, -WINDOW:]

# 长期销售水平与间歇性需求特征
long_features = pd.DataFrame({
    "id": sample_eval["id"].to_numpy(),
    "mean_56": recent_56.mean(axis=1),
    "zero_ratio_56": (recent_56 == 0).mean(axis=1),
    "nonzero_mean_56": np.divide(
        recent_56.sum(axis=1),
        (recent_56 > 0).sum(axis=1),
        out=np.zeros(len(sample_eval), dtype=np.float32),
        where=(recent_56 > 0).sum(axis=1) > 0
    ),
    "mean_28": sales_hist[:, -28:].mean(axis=1),
})

long_features["recent_vs_long_mean"] = (
    long_features["mean_28"] /
    (long_features["mean_56"] + 1e-6)
)

print("长期特征构造完成")
print("商品数：", len(long_features))
print("特征数：", long_features.shape[1] - 1)
print(long_features.head())
print("\n缺失值总数：", int(long_features.isna().sum().sum()))

# 保存特征，避免内核重启后丢失
long_features.to_csv(
    "/kaggle/working/long_sales_features_5000.csv",
    index=False
)
print("\n已保存：/kaggle/working/long_sales_features_5000.csv")

In [ ]:
import numpy as np
import pandas as pd

# 已有的预测误差
long_eval = long_features.copy()
long_eval["baseline_abs_error"] = np.abs(
    old_pred - actual
).mean(axis=1)
long_eval["baseline_sq_error"] = (
    (old_pred - actual) ** 2
).mean(axis=1)

# 按零销量比例分组，观察不同需求类型的误差
long_eval["需求类型"] = np.select(
    [
        long_eval["zero_ratio_56"] == 1,
        long_eval["zero_ratio_56"] >= 0.8
    ],
    [
        "近56天全零",
        "间歇性需求"
    ],
    default="其他商品"
)

summary = long_eval.groupby("需求类型").agg(
    商品数=("id", "count"),
    平均销量=("mean_56", "mean"),
    平均零销量比例=("zero_ratio_56", "mean"),
    平均非零日销量=("nonzero_mean_56", "mean"),
    平均绝对误差=("baseline_abs_error", "mean"),
    均方根误差=("baseline_sq_error", lambda x: np.sqrt(x.mean()))
).reset_index()

print(summary.to_string(index=False))

summary.to_csv(
    "/kaggle/working/long_sales_error_diagnostic.csv",
    index=False
)
print("\n诊断结果已保存。")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. 确认已有对象
required = ["sample_eval", "long_features", "calendar", "prices"]
missing = [name for name in required if name not in globals()]

if missing:
    raise NameError(
        f"缺少对象：{missing}。先恢复创建这些对象的代码单元格。"
    )

# 2. 确认商品 ID 唯一
assert sample_eval["id"].is_unique, "sample_eval 中存在重复商品 ID"
assert long_features["id"].is_unique, "long_features 中存在重复商品 ID"

# 3. 确认长期特征覆盖全部抽样商品
coverage = sample_eval[["id"]].merge(
    long_features[["id"]],
    on="id",
    how="left",
    indicator=True
)

print("抽样商品数：", len(sample_eval))
print("长期特征商品数：", len(long_features))
print("成功匹配：", (coverage["_merge"] == "both").sum())
print("未匹配：", (coverage["_merge"] == "left_only").sum())

# 4. 检查长期特征列
long_cols = [
    "mean_56",
    "zero_ratio_56",
    "nonzero_mean_56",
    "mean_28",
    "recent_vs_long_mean"
]

missing_cols = [c for c in long_cols if c not in long_features.columns]
if missing_cols:
    raise ValueError(f"长期特征缺少列：{missing_cols}")

# 5. 按商品 ID 合并，禁止依赖行号对齐
long_features_aligned = sample_eval[["id"]].merge(
    long_features[["id"] + long_cols],
    on="id",
    how="left",
    validate="one_to_one"
)

assert len(long_features_aligned) == len(sample_eval)
assert not long_features_aligned[long_cols].isna().any().any(), \
    "长期特征存在缺失值，需要先处理"

out_dir = Path("/kaggle/working/m5_checkpoints")
out_dir.mkdir(parents=True, exist_ok=True)

long_features_aligned.to_csv(
    out_dir / "long_features_aligned_5000.csv",
    index=False
)

print("\n长期特征已按商品 ID 对齐并保存。")
print("文件：", out_dir / "long_features_aligned_5000.csv")
print("\n特征预览：")
display(long_features_aligned.head())

In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# 1. 参数与输入
# ============================================================
N_ITEMS = 5000
TRAIN_START = 1551
TRAIN_END = 1913
VALID_START = 1914
VALID_END = 1941

OUT = Path("/kaggle/working/m5_checkpoints")
OUT.mkdir(parents=True, exist_ok=True)

assert "sample_eval" in globals()
assert "long_features" in globals()
assert "calendar" in globals()
assert "prices" in globals()

sales = sample_eval.copy().reset_index(drop=True)
long_feat = long_features.copy()

assert len(sales) == N_ITEMS
assert sales["id"].is_unique

day_cols = [f"d_{i}" for i in range(1, 1942)]
missing_days = [c for c in day_cols if c not in sales.columns]
if missing_days:
    raise ValueError(f"销量数据缺少日期列，例如：{missing_days[:5]}")

# ============================================================
# 2. 将商品 ID 拆成 item_id 和 store_id
# ============================================================
parts = sales["id"].str.replace(
    r"_evaluation$", "", regex=True
).str.split("_")

sales["item_id"] = parts.str[:-2].str.join("_")
sales["store_id"] = parts.str[-2:].str.join("_")

# 保证长期特征按商品 ID 对齐
long_cols = [
    "mean_56",
    "zero_ratio_56",
    "nonzero_mean_56",
    "mean_28",
    "recent_vs_long_mean",
]

long_feat = sales[["id"]].merge(
    long_feat[["id"] + long_cols],
    on="id",
    how="left",
    validate="one_to_one",
)

assert not long_feat[long_cols].isna().any().any()

# ============================================================
# 3. 准备日历和周价格
# ============================================================
cal = calendar.copy()
cal["d_num"] = cal["d"].str.extract(r"(\d+)").astype(int)

cal_features = cal[[
    "d_num", "weekday", "month", "year",
    "is_event", "wm_yr_wk"
]].copy()

cal_features["is_event"] = (
    cal_features["is_event"].fillna(0).astype(int)
)

price_table = prices[[
    "store_id", "item_id", "wm_yr_wk", "sell_price"
]].copy()

price_table = price_table.drop_duplicates(
    ["store_id", "item_id", "wm_yr_wk"]
)

# ============================================================
# 4. 重建带商品 ID 和目标日期的训练、验证数据
#    特征截止点为目标日期前 28 天
# ============================================================
sales_matrix = sales[day_cols].to_numpy(dtype=np.float32)
n = len(sales)

# 日历特征一次性准备
cal_by_day = cal_features.set_index("d_num")

def build_period(start_day, end_day, include_target=True):
    frames = []

    for target_day in range(start_day, end_day + 1):
        # 预测 target_day 时，最多只能使用 target_day-28 及以前的销量
        cutoff = target_day - 28

        frame = pd.DataFrame({
            "item_idx": np.arange(n, dtype=np.int32),
            "id": sales["id"].to_numpy(),
            "d_num": target_day,
            "target": (
                sales_matrix[:, target_day - 1]
                if include_target else np.zeros(n, dtype=np.float32)
            ),
            "lag_28": sales_matrix[:, cutoff - 1],
            "lag_56": sales_matrix[:, cutoff - 29],
            "lag_84": sales_matrix[:, cutoff - 57],
            "rolling_mean_7": sales_matrix[
                :, cutoff - 7:cutoff
            ].mean(axis=1),
            "rolling_mean_28": sales_matrix[
                :, cutoff - 28:cutoff
            ].mean(axis=1),
        })

        # 日历特征
        day_info = cal_by_day.loc[target_day]
        for c in ["weekday", "month", "year", "is_event", "wm_yr_wk"]:
            frame[c] = day_info[c]

        # SNAP 根据商品所属州匹配
        state = sales["id"].str.extract(
            r"_(CA|TX|WI)_\d+_evaluation$"
        )[0].to_numpy()

        snap_col = {
            "CA": "snap_CA",
            "TX": "snap_TX",
            "WI": "snap_WI",
        }

        # 仅对当前日期取出对应州的 SNAP 标记
        cal_row = cal.loc[cal["d_num"] == target_day].iloc[0]
        frame["snap"] = [
            cal_row[snap_col[s]] for s in state
        ]

        # 价格按商品、门店、周合并
        frame["item_id"] = sales["item_id"].to_numpy()
        frame["store_id"] = sales["store_id"].to_numpy()

        frame = frame.merge(
            price_table,
            on=["item_id", "store_id", "wm_yr_wk"],
            how="left",
            validate="many_to_one",
        )

        # 目标日期对应的价格；缺失价格用同商品最近可用价格补齐
        # 此处先保留缺失标记，再使用商品历史价格中位数填补
        frame["price_missing"] = frame["sell_price"].isna().astype("int8")
        frame["sell_price"] = frame["sell_price"].fillna(
            price_table.groupby(["item_id", "store_id"])["sell_price"]
            .transform("median")
            .reindex(frame.index)
        )

        frames.append(frame)

    result = pd.concat(frames, ignore_index=True)

    # 加入长期销量特征
    result = result.merge(
        long_feat,
        on="id",
        how="left",
        validate="many_to_one",
    )

    return result

print("正在构造训练集和验证集……")
train_df = build_period(TRAIN_START, TRAIN_END)
valid_df = build_period(VALID_START, VALID_END)

# ============================================================
# 5. 明确特征列表
# ============================================================
base_features = [
    "weekday", "month", "year", "is_event", "snap",
    "sell_price", "lag_28", "lag_56",
    "rolling_mean_7", "rolling_mean_28", "lag_84",
]

# 为了保持 11 特征基准不变，price_missing 暂不加入模型
long_features_list = long_cols

# 所有模型使用完全相同的行，防止样本不同造成不公平比较
train_df = train_df.dropna(
    subset=base_features + long_features_list + ["target"]
).copy()
valid_df = valid_df.dropna(
    subset=base_features + long_features_list + ["target"]
).copy()

X_train_base = train_df[base_features]
X_valid_base = valid_df[base_features]

X_train_long = train_df[base_features + long_features_list]
X_valid_long = valid_df[base_features + long_features_list]

y_train = train_df["target"].astype(np.float32)
y_valid = valid_df["target"].astype(np.float32)

print("训练集：", X_train_base.shape)
print("验证集：", X_valid_base.shape)
print("训练目标日期：", TRAIN_START, "至", TRAIN_END)
print("验证目标日期：", VALID_START, "至", VALID_END)

# ============================================================
# 6. 使用相同参数训练两个模型
# ============================================================
params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.2,
    "n_estimators": 350,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 80,
    "colsample_bytree": 0.8,
    "subsample": 0.8,
    "subsample_freq": 1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

print("\n开始训练基准模型（11 特征）……")
model_base_exp = lgb.LGBMRegressor(**params)
model_base_exp.fit(X_train_base, y_train)

print("开始训练长期销量模型（16 特征）……")
model_long_exp = lgb.LGBMRegressor(**params)
model_long_exp.fit(X_train_long, y_train)

pred_base = np.maximum(
    model_base_exp.predict(X_valid_base), 0
)
pred_long = np.maximum(
    model_long_exp.predict(X_valid_long), 0
)

# ============================================================
# 7. 整体指标比较
# ============================================================
def calc_metrics(y, pred):
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
    }

metrics_base = calc_metrics(y_valid, pred_base)
metrics_long = calc_metrics(y_valid, pred_long)

result = pd.DataFrame([
    {"model": "baseline_11", **metrics_base},
    {"model": "long_sales_16", **metrics_long},
])

result["MAE_change_vs_base"] = (
    result["MAE"] - metrics_base["MAE"]
)
result["RMSE_change_vs_base"] = (
    result["RMSE"] - metrics_base["RMSE"]
)

print("\n整体验证结果：")
display(result)

# ============================================================
# 8. 按需求类型分析
# ============================================================
valid_eval = valid_df[["id", "d_num", "target"]].copy()
valid_eval["pred_base"] = pred_base
valid_eval["pred_long"] = pred_long

# 每个商品只用截止 d_1913 的长期特征进行分组
valid_eval = valid_eval.merge(
    long_feat[["id", "zero_ratio_56", "mean_56"]],
    on="id",
    how="left",
    validate="many_to_one",
)

valid_eval["demand_group"] = np.select(
    [
        valid_eval["zero_ratio_56"] >= 0.95,
        valid_eval["zero_ratio_56"] >= 0.70,
    ],
    [
        "近56天几乎全零",
        "间歇性需求",
    ],
    default="其他商品",
)

valid_eval["abs_base"] = abs(
    valid_eval["target"] - valid_eval["pred_base"]
)
valid_eval["abs_long"] = abs(
    valid_eval["target"] - valid_eval["pred_long"]
)
valid_eval["sq_base"] = (
    valid_eval["target"] - valid_eval["pred_base"]
) ** 2
valid_eval["sq_long"] = (
    valid_eval["target"] - valid_eval["pred_long"]
) ** 2

group_result = valid_eval.groupby("demand_group").agg(
    rows=("target", "size"),
    MAE_base=("abs_base", "mean"),
    MAE_long=("abs_long", "mean"),
    MSE_base=("sq_base", "mean"),
    MSE_long=("sq_long", "mean"),
).reset_index()

group_result["RMSE_base"] = np.sqrt(group_result["MSE_base"])
group_result["RMSE_long"] = np.sqrt(group_result["MSE_long"])
group_result["MAE_change"] = (
    group_result["MAE_long"] - group_result["MAE_base"]
)
group_result["RMSE_change"] = (
    group_result["RMSE_long"] - group_result["RMSE_base"]
)

print("\n不同需求类型的验证结果：")
display(group_result)

# ============================================================
# 9. 保存结果和模型
# ============================================================
result.to_csv(
    OUT / "long_sales_model_comparison.csv",
    index=False
)
group_result.to_csv(
    OUT / "long_sales_group_comparison.csv",
    index=False
)

pd.DataFrame({
    "id": valid_df["id"],
    "d_num": valid_df["d_num"],
    "actual": y_valid,
    "pred_base": pred_base,
    "pred_long": pred_long,
}).to_csv(
    OUT / "long_sales_validation_predictions.csv",
    index=False
)

pd.DataFrame({
    "feature": X_train_long.columns,
    "importance_gain": model_long_exp.booster_.feature_importance(
        importance_type="gain"
    ),
    "importance_split": model_long_exp.booster_.feature_importance(
        importance_type="split"
    ),
}).sort_values(
    "importance_gain", ascending=False
).to_csv(
    OUT / "long_sales_feature_importance.csv",
    index=False
)

print("\n实验完成，结果保存至：", OUT)
print("1. long_sales_model_comparison.csv")
print("2. long_sales_group_comparison.csv")
print("3. long_sales_validation_predictions.csv")
print("4. long_sales_feature_importance.csv")


In [ ]:
import os
import pandas as pd

files = [
    "/kaggle/working/long_sales_features_5000.csv",
    "/kaggle/working/long_sales_error_diagnostic.csv",
    "/kaggle/working/comparison_10_vs_11_features.csv",
    "/kaggle/working/group_metrics_10_vs_11.csv"
]

for path in files:
    print("\n" + "=" * 60)
    print("文件：", os.path.basename(path))

    if not os.path.exists(path):
        print("不存在")
        continue

    df = pd.read_csv(path)
    print("形状：", df.shape)
    print("列名：", df.columns.tolist())
    print(df.head(3).to_string(index=False))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for name in files:
        if any(key in name.lower() for key in [
            "longterm",
            "long_term",
            "intermittent",
            "model",
            "importance",
            "result"
        ]):
            print(os.path.join(root, name))

In [ ]:
import os
import json
import pandas as pd

base = "/kaggle/working/m5_checkpoints"

for filename in [
    "longterm_experiment_plan.json",
    "long_sales_model_comparison.csv",
    "long_sales_feature_importance.csv"
]:
    path = os.path.join(base, filename)

    print("\n" + "=" * 65)
    print(filename)

    if not os.path.exists(path):
        print("文件不存在")
        continue

    if filename.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
    else:
        df = pd.read_csv(path)
        print("形状：", df.shape)
        print("列名：", df.columns.tolist())
        print(df.to_string(index=False))

In [ ]:
import os
import json
import pandas as pd

base = "/kaggle/working/m5_checkpoints"

# 1. 检查实验配置文件
path = os.path.join(base, "longterm_experiment_plan.json")

print("========== 实验配置文件 ==========")
if os.path.exists(path):
    print("文件大小：", os.path.getsize(path), "字节")

    with open(path, "r", encoding="utf-8-sig") as f:
        content = f.read()

    print("文件原始内容：", repr(content[:1000]))

    if content.strip():
        try:
            print("解析后的配置：")
            print(json.dumps(
                json.loads(content),
                ensure_ascii=False,
                indent=2
            ))
        except json.JSONDecodeError as e:
            print("JSON 格式错误：", e)
    else:
        print("判断：文件存在，但内容为空。")
else:
    print("文件不存在。")

# 2. 检查模型比较结果和特征重要性
for filename in [
    "long_sales_model_comparison.csv",
    "long_sales_feature_importance.csv"
]:
    path = os.path.join(base, filename)

    print("\n==========", filename, "==========")
    if not os.path.exists(path):
        print("文件不存在。")
        continue

    print("文件大小：", os.path.getsize(path), "字节")

    try:
        df = pd.read_csv(path)
        print("数据形状：", df.shape)
        print("列名：", df.columns.tolist())
        print(df.to_string(index=False))
    except Exception as e:
        print("读取失败：", repr(e))

# 3. 检查原模型文件是否存在
model_path = os.path.join(base, "model_11.pkl")
print("\n========== 原模型文件 ==========")
print("存在：", os.path.exists(model_path))
if os.path.exists(model_path):
    print("文件大小：", os.path.getsize(model_path), "字节")

In [ ]:
for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue
    try:
        if hasattr(obj, "booster_") and hasattr(obj, "predict"):
            print(
                name,
                "| 类型:", type(obj).__name__,
                "| 特征数:", getattr(obj, "n_features_in_", "未知")
            )
        elif name in ["model_11", "model_new", "model_longterm",
                      "long_sales_model", "features_11", "features_16"]:
            print(name, "| 类型:", type(obj).__name__)
    except Exception:
        pass
        

In [ ]:
import os
import json
import joblib
import pandas as pd

SAVE_DIR = "/kaggle/working/m5_handover"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. 保存两个模型
models = {
    "model_11": model_11,
    "model_long_exp": model_long_exp
}

for name, fitted_model in models.items():
    joblib.dump(
        fitted_model,
        os.path.join(SAVE_DIR, f"{name}.joblib")
    )

    fitted_model.booster_.save_model(
        os.path.join(SAVE_DIR, f"{name}.txt")
    )

    print(
        f"{name} 已保存，"
        f"特征数={fitted_model.n_features_in_}"
    )

# 2. 自动读取模型实际使用的特征名
feature_config = {
    name: list(fitted_model.booster_.feature_name())
    for name, fitted_model in models.items()
}

# 3. 保存实验配置
config = {
    "experiment": "M5 long-term sales features",
    "baseline_model": "model_11",
    "long_term_model": "model_long_exp",
    "feature_names": feature_config,
    "validation_days": list(range(1914, 1942)),
    "metrics_file": "long_sales_model_comparison.csv",
    "importance_file": "long_sales_feature_importance.csv"
}

with open(
    os.path.join(SAVE_DIR, "experiment_config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# 4. 复制已有实验结果
source_dir = "/kaggle/working/m5_checkpoints"

result_files = [
    "longterm_experiment_plan.json",
    "long_sales_model_comparison.csv",
    "long_sales_feature_importance.csv",
    "model_11.pkl"
]

for filename in result_files:
    source = os.path.join(source_dir, filename)

    if os.path.isfile(source) and os.path.getsize(source) > 0:
        import shutil
        shutil.copy2(
            source,
            os.path.join(SAVE_DIR, filename)
        )
        print("已复制：", filename)
    elif os.path.isfile(source):
        print("跳过空文件：", filename)

# 5. 检查保存结果
print("\n========== 保存文件清单 ==========")
for filename in sorted(os.listdir(SAVE_DIR)):
    path = os.path.join(SAVE_DIR, filename)
    print(f"{filename}: {os.path.getsize(path):,} bytes")

In [ ]:
import os

for folder in [
    "/kaggle/working",
    "/kaggle/working/m5_checkpoints",
    "/kaggle/working/m5_handover"
]:
    if os.path.isdir(folder):
        print(f"\n===== {folder} =====")
        for name in sorted(os.listdir(folder)):
            path = os.path.join(folder, name)
            if os.path.isfile(path):
                print(f"{os.path.getsize(path):>12,} bytes  {name}")

In [ ]:
print("prices 存在：", "prices" in globals())

if "prices" in globals():
    print("价格数据形状：", prices.shape)
    print("价格列：", prices.columns.tolist())
    print(prices.head(3))

print("\n长期特征模型：", "model_long_exp" in globals())
print("基准模型：", "model_11" in globals())
print("长期特征数量：", model_long_exp.n_features_in_)

In [ ]:
print("calendar 列：", calendar.columns.tolist())
print("calendar 日期映射示例：")
print(calendar[["d", "wm_yr_wk"]].head(3))

print("\nsample_eval：")
print("shape =", sample_eval.shape)
print(sample_eval.columns[:12].tolist())
print(sample_eval[["id", "item_id", "store_id"]].head(3))

print("\n长期特征表：")
print("long_features shape =", long_features.shape)
print(long_features.columns.tolist())
print(long_features.head(3))

print("\n模型训练特征数量：")
print("model_11 =", model_11.n_features_in_)
print("model_long_exp =", model_long_exp.n_features_in_)

print("\n模型特征名（如果模型保存了名称）：")
if hasattr(model_long_exp, "feature_name_"):
    print(model_long_exp.feature_name_)
else:
    print("当前模型没有可直接读取的 feature_name_ 属性")

In [ ]:

import numpy as np
import pandas as pd

# ==========================================
# 1. 提取 5000 个样本对应的价格历史
# ==========================================
item_store = sample_eval[["item_id", "store_id"]].drop_duplicates()

price_sub = prices.merge(
    item_store,
    on=["item_id", "store_id"],
    how="inner"
).copy()

price_sub = price_sub.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
).reset_index(drop=True)

keys = ["store_id", "item_id"]
g = price_sub.groupby(keys, sort=False)["sell_price"]

# ==========================================
# 2. 价格相对过去 28 天、56 天均价的变化比例
# 价格按周记录，因此使用过去 4 周、8 周近似
# ==========================================
price_sub["price_mean_4w"] = g.transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean()
)
price_sub["price_mean_8w"] = g.transform(
    lambda s: s.shift(1).rolling(8, min_periods=1).mean()
)

price_sub["price_vs_28d"] = (
    price_sub["sell_price"] / price_sub["price_mean_4w"] - 1
)
price_sub["price_vs_56d"] = (
    price_sub["sell_price"] / price_sub["price_mean_8w"] - 1
)

# ==========================================
# 3. 近期涨价 / 降价及幅度
# ==========================================
price_sub["price_prev"] = g.shift(1)
price_sub["price_change"] = (
    price_sub["sell_price"] / price_sub["price_prev"] - 1
)

price_sub["price_up"] = (
    price_sub["price_change"] > 0
).astype("int8")
price_sub["price_down"] = (
    price_sub["price_change"] < 0
).astype("int8")

# ==========================================
# 4. 价格波动程度
# 使用此前 4 周、8 周价格的标准差 / 均价
# ==========================================
price_sub["price_cv_4w"] = g.transform(
    lambda s: (
        s.shift(1).rolling(4, min_periods=2).std()
        / s.shift(1).rolling(4, min_periods=2).mean()
    )
)
price_sub["price_cv_8w"] = g.transform(
    lambda s: (
        s.shift(1).rolling(8, min_periods=2).std()
        / s.shift(1).rolling(8, min_periods=2).mean()
    )
)

# ==========================================
# 5. 历史需求频率
# 以预测起点 d_1913 为界，仅使用此前 56 天销量
# ==========================================
sales_cols = [f"d_{i}" for i in range(1858, 1914)]
sales_hist = sample_eval[["id", "item_id", "store_id"] + sales_cols].copy()

sales_values = sales_hist[sales_cols].to_numpy(dtype=np.float32)

sales_hist["demand_freq_28d"] = (
    sales_values[:, -28:] > 0
).mean(axis=1)

sales_hist["demand_freq_56d"] = (
    sales_values > 0
).mean(axis=1)

# 只用历史销量计算需求频率；按商品和门店匹配
demand_freq = sales_hist[
    ["item_id", "store_id", "demand_freq_28d", "demand_freq_56d"]
].drop_duplicates(keys)

# ==========================================
# 6. 取预测起点前最后一周的价格特征
# d_1913 对应的周作为历史价格特征的截止点
# ==========================================
cutoff_week = calendar.loc[
    calendar["d"].eq("d_1913"), "wm_yr_wk"
].iloc[0]

price_at_cutoff = price_sub[
    price_sub["wm_yr_wk"] <= cutoff_week
].sort_values("wm_yr_wk").groupby(
    keys, as_index=False
).tail(1).copy()

price_feature_cols = [
    "price_vs_28d",
    "price_vs_56d",
    "price_change",
    "price_up",
    "price_down",
    "price_cv_4w",
    "price_cv_8w"
]

price_features = price_at_cutoff[
    keys + price_feature_cols
].merge(
    demand_freq,
    on=keys,
    how="left"
)

# 价格变化 × 历史需求频率
price_features["price_change_x_freq28"] = (
    price_features["price_change"]
    * price_features["demand_freq_28d"]
)
price_features["price_change_x_freq56"] = (
    price_features["price_change"]
    * price_features["demand_freq_56d"]
)

print("价格特征表：", price_features.shape)
print("特征列：", price_features.columns.tolist())
print("\n缺失率：")
print(price_features.isna().mean())
print("\n特征示例：")
print(price_features.head())


In [ ]:
# 1. 检查价格特征整体分布
check_cols = [
    "price_vs_28d", "price_vs_56d",
    "price_change", "price_cv_4w", "price_cv_8w",
    "demand_freq_28d", "demand_freq_56d"
]

print("=== 价格与需求特征描述统计 ===")
display(price_features[check_cols].describe().T.round(4))

# 2. 检查涨价、降价、价格稳定的商品数量
print("\n=== 价格变化方向 ===")
print("涨价商品数：", (price_features["price_change"] > 0).sum())
print("降价商品数：", (price_features["price_change"] < 0).sum())
print("价格不变商品数：", (price_features["price_change"] == 0).sum())

# 3. 按需求频率分组，比较价格变化
price_features["demand_group"] = pd.cut(
    price_features["demand_freq_56d"],
    bins=[-0.001, 0, 0.25, 0.75, 1.001],
    labels=["零需求", "低频需求", "中频需求", "高频需求"]
)

print("\n=== 不同需求频率组的价格变化 ===")
display(
    price_features.groupby("demand_group", observed=False)
    .agg(
        商品数=("item_id", "size"),
        平均价格变化=("price_change", "mean"),
        平均28天相对价格=("price_vs_28d", "mean"),
        平均56天相对价格=("price_vs_56d", "mean"),
        平均价格波动=("price_cv_8w", "mean")
    )
    .round(4)
)

In [ ]:

# 快速检查：寻找现有训练集、验证集和标签
import pandas as pd
import numpy as np

keywords = ("train", "valid", "target", "label", "feature", "long")

for name, obj in list(globals().items()):
    if any(k in name.lower() for k in keywords):
        if isinstance(obj, pd.DataFrame):
            print(f"{name}: DataFrame {obj.shape}")
        elif isinstance(obj, pd.Series):
            print(f"{name}: Series {obj.shape}")
        elif isinstance(obj, np.ndarray):
            print(f"{name}: ndarray {obj.shape}")
        elif hasattr(obj, "n_features_in_"):
            print(f"{name}: 模型，特征数={obj.n_features_in_}")


In [ ]:
print("train_df 列名：")
print(train_df.columns.tolist())

print("\ntrain_df 前3行：")
display(train_df.head(3))

print("\nvalid_df 列名：")
print(valid_df.columns.tolist())

print("\nvalid_df 前3行：")
display(valid_df.head(3))

print("\n验证日期范围：")
if "d" in valid_df.columns:
    print(valid_df["d"].min(), "到", valid_df["d"].max())

In [ ]:

import numpy as np
import pandas as pd

# 1. 为每条训练/验证记录准备商品、门店、周信息
cal_week = calendar[["d", "wm_yr_wk"]].copy()
cal_week["d_num"] = cal_week["d"].str.replace("d_", "", regex=False).astype(int)

def attach_price_features(df):
    out = df.copy()

    # 确保周编号来自每条记录对应的日期
    if "wm_yr_wk" not in out.columns:
        out = out.merge(
            cal_week[["d_num", "wm_yr_wk"]],
            on="d_num",
            how="left",
            validate="many_to_one"
        )

    # 商品-门店-周的价格数据
    p = prices.sort_values(["store_id", "item_id", "wm_yr_wk"]).copy()
    keys = ["store_id", "item_id"]
    gp = p.groupby(keys, sort=False)["sell_price"]

    # 当前周价格变化及此前4周、8周价格均值
    p["price_prev"] = gp.shift(1)
    p["price_change"] = p["sell_price"] / p["price_prev"] - 1

    p["price_mean_4w"] = gp.transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean()
    )
    p["price_mean_8w"] = gp.transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean()
    )

    p["price_vs_28d"] = p["sell_price"] / p["price_mean_4w"] - 1
    p["price_vs_56d"] = p["sell_price"] / p["price_mean_8w"] - 1

    p["price_cv_4w"] = gp.transform(
        lambda s: s.shift(1).rolling(4, min_periods=2).std()
        / s.shift(1).rolling(4, min_periods=2).mean()
    )
    p["price_cv_8w"] = gp.transform(
        lambda s: s.shift(1).rolling(8, min_periods=2).std()
        / s.shift(1).rolling(8, min_periods=2).mean()
    )

    p["price_up"] = (p["price_change"] > 0).astype("int8")
    p["price_down"] = (p["price_change"] < 0).astype("int8")

    # 按商品、门店和周合并
    new_cols = [
        "store_id", "item_id", "wm_yr_wk",
        "price_vs_28d", "price_vs_56d", "price_change",
        "price_up", "price_down", "price_cv_4w", "price_cv_8w"
    ]

    out = out.drop(
        columns=[c for c in new_cols[3:] if c in out.columns],
        errors="ignore"
    ).merge(
        p[new_cols],
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one"
    )

    return out

train_price = attach_price_features(train_df)
valid_price = attach_price_features(valid_df)

# 2. 检查合并是否成功
price_cols = [
    "price_vs_28d", "price_vs_56d", "price_change",
    "price_up", "price_down", "price_cv_4w", "price_cv_8w"
]

print("训练集：", train_price.shape)
print("验证集：", valid_price.shape)
print("\n训练集价格特征缺失率：")
print(train_price[price_cols].isna().mean().round(4))
print("\n验证集价格特征缺失率：")
print(valid_price[price_cols].isna().mean().round(4))



In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 原模型使用的16个特征
base_features = list(model_long_exp.feature_name_)

# 新增7个价格特征
price_cols = [
    "price_vs_28d", "price_vs_56d", "price_change",
    "price_up", "price_down", "price_cv_4w", "price_cv_8w"
]

# 确认原有特征存在
missing_base = [
    c for c in base_features
    if c not in train_price.columns or c not in valid_price.columns
]
assert not missing_base, f"缺少原模型特征：{missing_base}"

# 缺失值处理：价格特征缺失时填0，并添加缺失标记
# 原始特征 price_missing 保留，不覆盖
for df in (train_price, valid_price):
    df[price_cols] = df[price_cols].replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)

new_features = base_features + price_cols

X_train_price = train_price[new_features].astype("float32")
y_train_price = train_price["target"].astype("float32")

X_valid_price = valid_price[new_features].astype("float32")
y_valid_price = valid_price["target"].astype("float32")

# 保留原有模型，使用完全相同的验证集
pred_base = model_long_exp.predict(
    valid_df[base_features]
)
pred_price_old = model_long_exp.predict(
    X_valid_price[base_features]
)

# 新模型：沿用现有模型的大致复杂度
model_price = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_price.fit(X_train_price, y_train_price)

pred_price = np.maximum(
    model_price.predict(X_valid_price), 0
)

# 对齐验证标签与原模型预测
y_true = y_valid_price.to_numpy()

results = pd.DataFrame([
    {
        "model": "long_sales_16",
        "MAE": mean_absolute_error(y_true, pred_base),
        "RMSE": mean_squared_error(y_true, pred_base) ** 0.5
    },
    {
        "model": "price_enhanced_23",
        "MAE": mean_absolute_error(y_true, pred_price),
        "RMSE": mean_squared_error(y_true, pred_price) ** 0.5
    }
])

print("模型对比：")
display(results.round(6))

base_mae = results.loc[0, "MAE"]
new_mae = results.loc[1, "MAE"]
base_rmse = results.loc[0, "RMSE"]
new_rmse = results.loc[1, "RMSE"]

print(f"MAE变化：{(new_mae / base_mae - 1) * 100:+.2f}%")
print(f"RMSE变化：{(new_rmse / base_rmse - 1) * 100:+.2f}%")
print("负数表示误差下降，正数表示误差上升。")

# 保存实验结果
results.to_csv(
    "/kaggle/working/price_model_comparison.csv",
    index=False
)

model_price.booster_.save_model(
    "/kaggle/working/m5_price_model.txt"
)

print("\n结果已保存到 /kaggle/working/")


In [ ]:
# 查看当前环境中可能相关的模型和数据变量
for name, obj in list(globals().items()):
    if any(key in name.lower() for key in [
        "price", "model", "valid", "pred"
    ]):
        if hasattr(obj, "feature_name_"):
            print(f"模型：{name}，特征数：{len(obj.feature_name_)}")
        elif hasattr(obj, "shape"):
            print(f"数据：{name}，shape={obj.shape}")
        elif isinstance(obj, (list, tuple)):
            print(f"变量：{name}，类型={type(obj).__name__}，长度={len(obj)}")

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

# 1. 使用当前真实存在的模型和验证集
model = model_price
X = X_valid_price[model.feature_name_].copy()
y = np.asarray(y_valid_price)

# 2. 计算 Gain 和 Split
result = pd.DataFrame({
    "feature": model.feature_name_,
    "gain": model.booster_.feature_importance(importance_type="gain"),
    "split": model.booster_.feature_importance(importance_type="split")
})

result["gain_pct"] = result["gain"] / result["gain"].sum() * 100

# 3. 计算验证集基准 MAE
baseline_pred = np.maximum(model.predict(X), 0)
baseline_mae = mean_absolute_error(y, baseline_pred)

# 4. 逐个打乱特征，观察 MAE 的变化
rng = np.random.default_rng(42)
perm_results = []

for col in X.columns:
    X_perm = X.copy()
    X_perm[col] = rng.permutation(X_perm[col].to_numpy())

    pred_perm = np.maximum(model.predict(X_perm), 0)
    perm_mae = mean_absolute_error(y, pred_perm)

    perm_results.append({
        "feature": col,
        "permutation_mae_increase": perm_mae - baseline_mae
    })

result = result.merge(pd.DataFrame(perm_results), on="feature")
result = result.sort_values("gain", ascending=False).reset_index(drop=True)

print(f"验证集基准 MAE：{baseline_mae:.6f}")
print("\n全部特征重要性：")
display(result.round(6))

# 5. 单独查看价格特征
price_cols = [
    "price_vs_28d", "price_vs_56d", "price_change",
    "price_up", "price_down", "price_cv_4w", "price_cv_8w"
]

price_result = result[result["feature"].isin(price_cols)].copy()

print("\n新增价格特征重要性：")
display(price_result.sort_values("gain", ascending=False).round(6))

# 6. 画图
plot_df = price_result.sort_values("gain", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["feature"], plot_df["gain"])
plt.xlabel("Total Gain")
plt.title("Price Feature Importance")
plt.tight_layout()
plt.show()

# 7. 保存
result.to_csv(
    "/kaggle/working/price_feature_importance.csv",
    index=False
)

print("结果已保存：/kaggle/working/price_feature_importance.csv")


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# A：原来的16特征模型
base_features = list(model_long_exp.feature_name_)

# B：当前23特征模型
all_features = list(model_price.feature_name_)

# C：保留3个有一定正向信号的价格特征
keep_price = [
    "price_vs_56d",
    "price_vs_28d",
    "price_cv_8w"
]
reduced_features = [
    c for c in all_features
    if c not in [
        "price_change", "price_up", "price_down", "price_cv_4w"
    ]
]

# 确保使用相同的训练集和验证集
y_train_exp = train_price["target"].astype("float32")
y_valid_exp = valid_price["target"].astype("float32")

def evaluate_model(name, model, X, y):
    pred = np.maximum(model.predict(X), 0)
    return {
        "model": name,
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred))
    }

# A：基准模型已有预测
pred_A = np.maximum(
    model_long_exp.predict(valid_price[base_features]), 0
)

# B：现有价格增强模型
pred_B = np.maximum(
    model_price.predict(valid_price[all_features]), 0
)

# C：重新训练删减后的价格模型
model_price_reduced = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_price_reduced.fit(
    train_price[reduced_features],
    y_train_exp
)

pred_C = np.maximum(
    model_price_reduced.predict(valid_price[reduced_features]), 0
)

results = pd.DataFrame([
    {
        "model": "A_long_sales_16",
        "MAE": mean_absolute_error(y_valid_exp, pred_A),
        "RMSE": np.sqrt(mean_squared_error(y_valid_exp, pred_A))
    },
    {
        "model": "B_price_enhanced_23",
        "MAE": mean_absolute_error(y_valid_exp, pred_B),
        "RMSE": np.sqrt(mean_squared_error(y_valid_exp, pred_B))
    },
    {
        "model": "C_price_reduced",
        "MAE": mean_absolute_error(y_valid_exp, pred_C),
        "RMSE": np.sqrt(mean_squared_error(y_valid_exp, pred_C))
    }
])

print("三组模型对比：")
display(results.round(6))

results.to_csv(
    "/kaggle/working/price_model_comparison.csv",
    index=False
)

print("对比结果已保存到 /kaggle/working/price_model_comparison.csv")


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 固定同一套23特征和训练/验证数据
features = list(model_price.feature_name_)

Xtr = train_price[features]
Xva = valid_price[features]
ytr = train_price["target"].astype("float32")
yva = valid_price["target"].astype("float32")

results = []
tuned_models = {}

for leaves in [15, 31, 63]:
    print(f"训练 num_leaves={leaves} ...")

    m = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=leaves,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    m.fit(Xtr, ytr)
    pred = np.maximum(m.predict(Xva), 0)

    results.append({
        "num_leaves": leaves,
        "MAE": mean_absolute_error(yva, pred),
        "RMSE": np.sqrt(mean_squared_error(yva, pred))
    })
    tuned_models[leaves] = m

results_df = pd.DataFrame(results)
print("\n参数实验结果：")
display(results_df.round(6))

results_df.to_csv(
    "/kaggle/working/num_leaves_comparison.csv",
    index=False
)
print("结果已保存至 /kaggle/working/num_leaves_comparison.csv")


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

features = list(model_price.feature_name_)

Xtr = train_price[features]
Xva = valid_price[features]
ytr = train_price["target"].astype("float32")
yva = valid_price["target"].astype("float32")

experiments = [
    {"name": "A_baseline", "lr": 0.05, "n": 500},
    {"name": "B_smaller_lr", "lr": 0.03, "n": 800},
    {"name": "C_smallest_lr", "lr": 0.02, "n": 1200},
]

results = []
lr_models = {}

for exp in experiments:
    print(f"训练 {exp['name']} ...")

    m = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=exp["n"],
        learning_rate=exp["lr"],
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    m.fit(Xtr, ytr)
    pred = np.maximum(m.predict(Xva), 0)

    results.append({
        "model": exp["name"],
        "learning_rate": exp["lr"],
        "n_estimators": exp["n"],
        "MAE": mean_absolute_error(yva, pred),
        "RMSE": np.sqrt(mean_squared_error(yva, pred))
    })

    lr_models[exp["name"]] = m

results_lr = pd.DataFrame(results)
display(results_lr.round(6))

results_lr.to_csv(
    "/kaggle/working/learning_rate_comparison.csv",
    index=False
)
print("结果已保存到 /kaggle/working/learning_rate_comparison.csv")


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

features = list(model_price.feature_name_)
Xtr = train_price[features]
Xva = valid_price[features]
ytr = train_price["target"].astype("float32")
yva = valid_price["target"].astype("float32")

results = []
models_rounds = {}

for n in [1200, 1600, 2000]:
    print(f"训练轮数：{n}")

    m = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=n,
        learning_rate=0.02,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    m.fit(Xtr, ytr)
    pred = np.maximum(m.predict(Xva), 0)

    results.append({
        "n_estimators": n,
        "MAE": mean_absolute_error(yva, pred),
        "RMSE": np.sqrt(mean_squared_error(yva, pred))
    })
    models_rounds[n] = m

results_rounds = pd.DataFrame(results)
display(results_rounds.round(6))

results_rounds.to_csv(
    "/kaggle/working/rounds_comparison.csv",
    index=False
)
print("结果已保存到 /kaggle/working/rounds_comparison.csv")


In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

features = list(model_price.feature_name_)

Xtr = train_price[features]
Xva = valid_price[features]
ytr = train_price["target"].astype("float32")
yva = valid_price["target"].astype("float32")

model_early = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=3000,       # 设置足够大的上限
    learning_rate=0.02,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_early.fit(
    Xtr,
    ytr,
    eval_set=[(Xva, yva)],
    eval_metric="l1",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

pred_early = np.maximum(
    model_early.predict(Xva, num_iteration=model_early.best_iteration_),
    0
)

result_early = pd.DataFrame([{
    "best_iteration": model_early.best_iteration_,
    "MAE": mean_absolute_error(yva, pred_early),
    "RMSE": np.sqrt(mean_squared_error(yva, pred_early))
}])

display(result_early.round(6))

result_early.to_csv(
    "/kaggle/working/early_stopping_result.csv",
    index=False
)

print("结果已保存至 /kaggle/working/early_stopping_result.csv")


In [ ]:
print("训练集形状：", train_price.shape)
print("验证集形状：", valid_price.shape)

print("\n训练目标统计：")
print(train_price["target"].describe())

print("\n验证目标统计：")
print(valid_price["target"].describe())

print("\n特征数：")
print("model_price:", len(model_price.feature_name_))
print("model_early:", len(model_early.feature_name_))

print("\n目标是否包含负数：")
print("train:", (train_price["target"] < 0).sum())
print("valid:", (valid_price["target"] < 0).sum())

print("\n验证集预测对比：")
print("model_price MAE:", 
      np.mean(np.abs(y_valid_price - np.maximum(
          model_price.predict(X_valid_price[model_price.feature_name_]), 0
      ))))
print("model_early MAE:", 
      np.mean(np.abs(y_valid_price - np.maximum(
          model_early.predict(X_valid_price[model_early.feature_name_]), 0
      ))))

In [ ]:
import pandas as pd

for name in ["sample_eval", "sales", "sales_train_validation",
             "sales_train_evaluation", "calendar", "prices"]:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: shape={obj.shape}")
        print("列名示例：", obj.columns[:8].tolist())

In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# 0. 检查必要数据
# ============================================================
required = ["sales", "calendar", "train_df", "valid_df", "model_long_exp"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"缺少必要变量：{missing}")

base_features = list(model_long_exp.booster_.feature_name())
forecast_horizon = 28

# 建立商品序列与原始销量矩阵的对应关系
daily_cols = [
    c for c in sales.columns
    if c.startswith("d_") and c[2:].isdigit()
]
daily_cols = sorted(daily_cols, key=lambda c: int(c[2:]))

sales_ids = sales["id"].astype(str).to_numpy()
id_to_row = {sid: i for i, sid in enumerate(sales_ids)}

sales_matrix = sales[daily_cols].to_numpy(dtype=np.float32)
max_day = sales_matrix.shape[1]

print("原始销量矩阵：", sales_matrix.shape)
print("基准特征数：", len(base_features))

# ============================================================
# 1. 根据每条记录的历史销量构造候选特征
#    注意：不添加 lag_7、lag_14、lag_21
# ============================================================
def add_candidate_features(df):
    out = df.copy()

    row_idx = out["id"].astype(str).map(id_to_row)
    if row_idx.isna().any():
        raise ValueError("有 id 无法与 sales 匹配，请检查两张表的 id 格式。")

    row_idx = row_idx.to_numpy(dtype=np.int32)
    day_idx = out["d_num"].to_numpy(dtype=np.int32)

    if day_idx.min() < 1 or day_idx.max() > max_day:
        raise ValueError("d_num 超出原始销量数据覆盖范围。")

    # 在预测日 t，只使用截至 t-28 的历史销量。
    # 这样 train 和 valid 使用相同的信息截止规则。
    cutoff = day_idx - forecast_horizon

    def history_values(offset):
        # offset=28 表示取 t-28；offset=364 表示取 t-364
        pos = day_idx - offset
        result = np.zeros(len(out), dtype=np.float32)
        ok = pos >= 1
        result[ok] = sales_matrix[row_idx[ok], pos[ok] - 1]
        return result

    out["lag_364"] = history_values(364)

    # 滚动窗口从 t-28 开始向过去取值，避免使用预测期销量
    for window in [7, 28, 56, 364]:
        values = np.zeros(len(out), dtype=np.float32)
        std_values = np.zeros(len(out), dtype=np.float32)
        zero_values = np.zeros(len(out), dtype=np.float32)

        for i, (r, c) in enumerate(zip(row_idx, cutoff)):
            end = min(c, max_day)
            start = max(1, end - window + 1)

            if end < 1:
                continue

            hist = sales_matrix[r, start - 1:end]
            if len(hist) == 0:
                continue

            values[i] = np.mean(hist)
            std_values[i] = np.std(hist)
            zero_values[i] = np.mean(hist == 0)

        out[f"hist_mean_{window}"] = values
        out[f"hist_std_{window}"] = std_values
        out[f"hist_zero_ratio_{window}"] = zero_values

    # 日历特征：从 calendar 的 d 列映射到预测日期
    cal = calendar.copy()
    cal["d_num"] = (
        cal["d"].astype(str).str.replace("d_", "", regex=False).astype(int)
    )
    cal = cal.drop_duplicates("d_num").set_index("d_num")

    out["weekday_num"] = out["weekday"].map({
        "Monday": 0, "Tuesday": 1, "Wednesday": 2,
        "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
    }).fillna(0).astype(np.int8)

    out["day_of_month"] = (
        out["d_num"].map(cal["date"])
        .pipe(pd.to_datetime, errors="coerce")
        .dt.day.fillna(1).astype(np.int8)
    )

    out["is_weekend"] = out["weekday_num"].isin([5, 6]).astype(np.int8)

    # 事件前后 3 天：使用已知日历信息，不使用未来销量
    event_day = (
        cal["event_name_1"].notna() | cal["event_name_2"].notna()
    )
    event_days = cal.index[event_day].to_numpy()

    def near_event(day):
        if len(event_days) == 0:
            return np.zeros(len(day), dtype=np.int8)
        return np.array([
            int(np.any(np.abs(event_days - d) <= 3))
            for d in day
        ], dtype=np.int8)

    out["near_event_3d"] = near_event(day_idx)

    # 价格特征：如果之前已有 sell_price，就构造相对价格指标
    if "sell_price" in out.columns:
        price = pd.to_numeric(out["sell_price"], errors="coerce")
        out["price_log1p"] = np.log1p(price.clip(lower=0))
        out["price_missing_new"] = price.isna().astype(np.int8)

    # 近期需求趋势：短窗口与长窗口之比
    out["hist_mean_ratio_28_364"] = (
        out["hist_mean_28"] / (out["hist_mean_364"] + 1e-3)
    ).clip(0, 20)

    return out


train_candidate = add_candidate_features(train_df)
valid_candidate = add_candidate_features(valid_df)

# ============================================================
# 2. 自动挑选可用的新增数值特征
# ============================================================
new_features = [
    c for c in train_candidate.columns
    if c.startswith((
        "lag_364", "hist_mean_", "hist_std_", "hist_zero_ratio_",
        "weekday_num", "day_of_month", "is_weekend",
        "near_event_3d", "price_log1p", "price_missing_new",
        "hist_mean_ratio_"
    ))
]

# 排除基准中已有的字段，避免重复
new_features = [c for c in new_features if c not in base_features]
candidate_features = base_features + new_features

# 仅保留两张表都存在且有变化的特征
candidate_features = [
    c for c in candidate_features
    if c in train_candidate.columns
    and c in valid_candidate.columns
    and train_candidate[c].nunique(dropna=False) > 1
]

Xtr = train_candidate[candidate_features].replace([np.inf, -np.inf], np.nan)
Xva = valid_candidate[candidate_features].replace([np.inf, -np.inf], np.nan)
ytr = train_candidate["target"].astype(np.float32)
yva = valid_candidate["target"].astype(np.float32)

# ============================================================
# 3. 训练候选模型并比较基准
# ============================================================
model_candidate = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1600,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_candidate.fit(Xtr, ytr)

pred_candidate = np.maximum(model_candidate.predict(Xva), 0)

# 基准模型使用之前保存的验证预测
if "pred_long" in globals() and len(pred_long) == len(yva):
    pred_base = np.maximum(np.asarray(pred_long), 0)
elif "valid_pred" in globals() and len(valid_pred) == len(yva):
    pred_base = np.maximum(np.asarray(valid_pred), 0)
else:
    pred_base = None

def report_metrics(name, y, pred):
    mae = np.mean(np.abs(y - pred))
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    return {"model": name, "MAE": mae, "RMSE": rmse}

results = [
    report_metrics("候选特征模型", yva.to_numpy(), pred_candidate)
]

if pred_base is not None:
    results.insert(
        0, report_metrics("现有基准预测", yva.to_numpy(), pred_base)
    )

results_df = pd.DataFrame(results)
print("\n========== 验证集指标 ==========")
print(results_df.to_string(index=False))

# ============================================================
# 4. 特征重要性
# ============================================================
importance_df = pd.DataFrame({
    "feature": candidate_features,
    "gain": model_candidate.booster_.feature_importance(importance_type="gain"),
    "split": model_candidate.booster_.feature_importance(importance_type="split")
}).sort_values("gain", ascending=False)

print("\n========== Gain 排名前 20 ==========")
print(importance_df.head(20).to_string(index=False))

# ============================================================
# 5. 按需求稀疏程度检查误差
# ============================================================
if "zero_ratio_56" in valid_candidate.columns:
    diagnostics = valid_candidate[["zero_ratio_56", "target"]].copy()
    diagnostics["pred"] = pred_candidate
    diagnostics["abs_error"] = np.abs(
        diagnostics["target"] - diagnostics["pred"]
    )
    diagnostics["demand_group"] = pd.cut(
        diagnostics["zero_ratio_56"],
        bins=[-0.001, 0.25, 0.75, 1.001],
        labels=["较连续", "间歇性", "高度稀疏"]
    )
    group_report = diagnostics.groupby(
        "demand_group", observed=False
    ).agg(
        rows=("target", "size"),
        actual_mean=("target", "mean"),
        pred_mean=("pred", "mean"),
        MAE=("abs_error", "mean")
    ).reset_index()

    print("\n========== 按需求稀疏程度的误差 ==========")
    print(group_report.to_string(index=False))

print("\n本轮新增特征数：", len(new_features))
print("新增特征：", new_features)
print("完成：候选模型已训练，基准与候选模型指标已输出。")


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

SAVE_DIR = "/kaggle/working/m5_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# 检查候选模型与基准模型
for name in ["model_long_exp", "model_11", "sales", "calendar", "prices"]:
    obj = globals().get(name)
    print(
        name, "：",
        "存在" if obj is not None else "不存在",
        "| 类型：" + type(obj).__name__ if obj is not None else ""
    )

# 保存已有的候选模型
if "model_long_exp" in globals():
    joblib.dump(
        model_long_exp,
        os.path.join(SAVE_DIR, "model_long_exp_latest.joblib")
    )
    print("候选模型已保存")

    try:
        feature_names = list(model_long_exp.booster_.feature_name())
    except Exception:
        feature_names = list(
            getattr(model_long_exp, "feature_name_", [])
        )

    if feature_names:
        with open(
            os.path.join(SAVE_DIR, "candidate_features.json"),
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(feature_names, f, ensure_ascii=False, indent=2)
        print("特征数量：", len(feature_names))
        print("特征列表：", feature_names)

print("\n检查完成，保存目录：", SAVE_DIR)

In [ ]:
import os
import json
import joblib

SAVE_DIR = "/kaggle/working/m5_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# 查找当前 Notebook 中所有已经训练的 LightGBM 模型
candidates = []

for name, obj in list(globals().items()):
    try:
        if hasattr(obj, "booster_"):
            features = list(obj.booster_.feature_name())
            if len(features) == 36:
                candidates.append((name, obj, features))
    except Exception:
        pass

if not candidates:
    print("没有找到已训练的 36 特征模型。")
    print("请不要重新训练；把刚才候选模型训练代码中最后的模型变量名发给我。")
else:
    # 优先使用最近实验中常见的变量名
    preferred = [
        "model_candidate", "candidate_model",
        "model_new_features", "model_36",
        "model_candidate_36", "model"
    ]

    candidates.sort(
        key=lambda x: preferred.index(x[0])
        if x[0] in preferred else len(preferred)
    )

    model_name, model_36, features_36 = candidates[0]

    joblib_path = os.path.join(
        SAVE_DIR, "model_36_features.joblib"
    )
    txt_path = os.path.join(
        SAVE_DIR, "model_36_features.txt"
    )
    json_path = os.path.join(
        SAVE_DIR, "model_36_features.json"
    )

    joblib.dump(model_36, joblib_path)
    model_36.booster_.save_model(txt_path)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(features_36, f, ensure_ascii=False, indent=2)

    # 检查保存文件
    loaded_model = joblib.load(joblib_path)

    print("保存成功！")
    print("原模型变量名：", model_name)
    print("特征数量：", len(features_36))
    print("验证模型特征数：", len(loaded_model.booster_.feature_name()))
    print("Joblib：", joblib_path)
    print("LightGBM 原生模型：", txt_path)
    print("特征清单：", json_path)

In [ ]:
import lightgbm as lgb
from pathlib import Path
import joblib
import json

OUT = Path("/kaggle/working/model_backups")
OUT.mkdir(parents=True, exist_ok=True)

saved = []

for name, obj in list(globals().items()):
    if not isinstance(obj, lgb.LGBMRegressor):
        continue

    try:
        n = obj.n_features_in_
        booster = obj.booster_
    except Exception:
        continue

    # 保存模型和特征名
    model_path = OUT / f"{name}.txt"
    booster.save_model(str(model_path))

    features = list(booster.feature_name())
    meta_path = OUT / f"{name}_features.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(
            {"model_name": name, "n_features": int(n),
             "features": features},
            f, ensure_ascii=False, indent=2
        )

    saved.append((name, n, str(model_path)))

print("保存完成：")
for name, n, path in saved:
    print(f"{name}: {n} 个特征 -> {path}")

if not saved:
    print("当前内存中没有可保存的已训练 LightGBM 模型。")

In [ ]:
from pathlib import Path

root = Path("/kaggle/working")

keywords = (
    "model", "candidate", "feature", "importance",
    "ablation", "comparison", "result", "36"
)

files = [
    p for p in root.rglob("*")
    if p.is_file()
    and any(k in p.name.lower() for k in keywords)
    and p.suffix.lower() in {
        ".txt", ".pkl", ".joblib", ".json", ".csv", ".parquet"
    }
]

print(f"找到 {len(files)} 个相关文件：")
for p in files:
    print(f"{p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
import json
import lightgbm as lgb
from pathlib import Path

root = Path("/kaggle/working")

# 1. 查看候选特征清单
p = root / "m5_checkpoints/candidate_features.json"
print("=== 候选特征清单 ===")
if p.exists():
    print(p.read_text(encoding="utf-8"))
else:
    print("文件不存在")

# 2. 查看已有 LightGBM 模型的特征数量
print("\n=== 已保存模型的特征数 ===")
for p in root.rglob("*.txt"):
    try:
        model = lgb.Booster(model_file=str(p))
        print(f"{p.name}: {model.num_feature()} 个特征")
        if model.num_feature() == 36:
            print("  特征名：", model.feature_name())
    except Exception:
        pass

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/kaggle/working")

for name in [
    "feature_ablation_results.csv",
    "importance_14_features.csv",
    "m5_checkpoints/long_sales_feature_importance.csv",
    "m5_checkpoints/long_sales_model_comparison.csv",
]:
    path = root / name
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    if path.exists():
        df = pd.read_csv(path)
        print("形状：", df.shape)
        print(df.to_string(index=False))
    else:
        print("文件不存在")

In [ ]:
from pathlib import Path
import json
import lightgbm as lgb

out = Path("/kaggle/working/m5_checkpoints")
out.mkdir(parents=True, exist_ok=True)

if "model_candidate" not in globals():
    print("model_candidate 已不在当前内存中，未进行训练。")
else:
    model = globals()["model_candidate"]

    if not isinstance(model, lgb.LGBMRegressor):
        print("model_candidate 不是预期的 LightGBM 模型。")
    else:
        booster = model.booster_
        features = list(booster.feature_name())

        booster.save_model(str(out / "model_candidate.txt"))

        with open(out / "model_candidate_features.json", "w",
                  encoding="utf-8") as f:
            json.dump(features, f, ensure_ascii=False, indent=2)

        print("模型已保存")
        print("实际特征数：", booster.num_feature())
        print("模型路径：", out / "model_candidate.txt")
        print("特征清单：", out / "model_candidate_features.json")

In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb

# ============================================================
# 0. 检查必要数据
# ============================================================
required = ["sales", "calendar", "train_df", "valid_df", "model_long_exp"]
missing = [x for x in required if x not in globals()]
if missing:
    raise ValueError(f"缺少必要变量：{missing}")

base_features = list(model_long_exp.booster_.feature_name())
forecast_horizon = 28

# 建立商品序列与原始销量矩阵的对应关系
daily_cols = [
    c for c in sales.columns
    if c.startswith("d_") and c[2:].isdigit()
]
daily_cols = sorted(daily_cols, key=lambda c: int(c[2:]))

sales_ids = sales["id"].astype(str).to_numpy()
id_to_row = {sid: i for i, sid in enumerate(sales_ids)}

sales_matrix = sales[daily_cols].to_numpy(dtype=np.float32)
max_day = sales_matrix.shape[1]

print("原始销量矩阵：", sales_matrix.shape)
print("基准特征数：", len(base_features))

# ============================================================
# 1. 根据每条记录的历史销量构造候选特征
#    注意：不添加 lag_7、lag_14、lag_21
# ============================================================
def add_candidate_features(df):
    out = df.copy()

    row_idx = out["id"].astype(str).map(id_to_row)
    if row_idx.isna().any():
        raise ValueError("有 id 无法与 sales 匹配，请检查两张表的 id 格式。")

    row_idx = row_idx.to_numpy(dtype=np.int32)
    day_idx = out["d_num"].to_numpy(dtype=np.int32)

    if day_idx.min() < 1 or day_idx.max() > max_day:
        raise ValueError("d_num 超出原始销量数据覆盖范围。")

    # 在预测日 t，只使用截至 t-28 的历史销量。
    # 这样 train 和 valid 使用相同的信息截止规则。
    cutoff = day_idx - forecast_horizon

    def history_values(offset):
        # offset=28 表示取 t-28；offset=364 表示取 t-364
        pos = day_idx - offset
        result = np.zeros(len(out), dtype=np.float32)
        ok = pos >= 1
        result[ok] = sales_matrix[row_idx[ok], pos[ok] - 1]
        return result

    out["lag_364"] = history_values(364)

    # 滚动窗口从 t-28 开始向过去取值，避免使用预测期销量
    for window in [7, 28, 56, 364]:
        values = np.zeros(len(out), dtype=np.float32)
        std_values = np.zeros(len(out), dtype=np.float32)
        zero_values = np.zeros(len(out), dtype=np.float32)

        for i, (r, c) in enumerate(zip(row_idx, cutoff)):
            end = min(c, max_day)
            start = max(1, end - window + 1)

            if end < 1:
                continue

            hist = sales_matrix[r, start - 1:end]
            if len(hist) == 0:
                continue

            values[i] = np.mean(hist)
            std_values[i] = np.std(hist)
            zero_values[i] = np.mean(hist == 0)

        out[f"hist_mean_{window}"] = values
        out[f"hist_std_{window}"] = std_values
        out[f"hist_zero_ratio_{window}"] = zero_values

    # 日历特征：从 calendar 的 d 列映射到预测日期
    cal = calendar.copy()
    cal["d_num"] = (
        cal["d"].astype(str).str.replace("d_", "", regex=False).astype(int)
    )
    cal = cal.drop_duplicates("d_num").set_index("d_num")

    out["weekday_num"] = out["weekday"].map({
        "Monday": 0, "Tuesday": 1, "Wednesday": 2,
        "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
    }).fillna(0).astype(np.int8)

    out["day_of_month"] = (
        out["d_num"].map(cal["date"])
        .pipe(pd.to_datetime, errors="coerce")
        .dt.day.fillna(1).astype(np.int8)
    )

    out["is_weekend"] = out["weekday_num"].isin([5, 6]).astype(np.int8)

    # 事件前后 3 天：使用已知日历信息，不使用未来销量
    event_day = (
        cal["event_name_1"].notna() | cal["event_name_2"].notna()
    )
    event_days = cal.index[event_day].to_numpy()

    def near_event(day):
        if len(event_days) == 0:
            return np.zeros(len(day), dtype=np.int8)
        return np.array([
            int(np.any(np.abs(event_days - d) <= 3))
            for d in day
        ], dtype=np.int8)

    out["near_event_3d"] = near_event(day_idx)

    # 价格特征：如果之前已有 sell_price，就构造相对价格指标
    if "sell_price" in out.columns:
        price = pd.to_numeric(out["sell_price"], errors="coerce")
        out["price_log1p"] = np.log1p(price.clip(lower=0))
        out["price_missing_new"] = price.isna().astype(np.int8)

    # 近期需求趋势：短窗口与长窗口之比
    out["hist_mean_ratio_28_364"] = (
        out["hist_mean_28"] / (out["hist_mean_364"] + 1e-3)
    ).clip(0, 20)

    return out


train_candidate = add_candidate_features(train_df)
valid_candidate = add_candidate_features(valid_df)

# ============================================================
# 2. 自动挑选可用的新增数值特征
# ============================================================
new_features = [
    c for c in train_candidate.columns
    if c.startswith((
        "lag_364", "hist_mean_", "hist_std_", "hist_zero_ratio_",
        "weekday_num", "day_of_month", "is_weekend",
        "near_event_3d", "price_log1p", "price_missing_new",
        "hist_mean_ratio_"
    ))
]

# 排除基准中已有的字段，避免重复
new_features = [c for c in new_features if c not in base_features]
candidate_features = base_features + new_features

# 仅保留两张表都存在且有变化的特征
candidate_features = [
    c for c in candidate_features
    if c in train_candidate.columns
    and c in valid_candidate.columns
    and train_candidate[c].nunique(dropna=False) > 1
]

Xtr = train_candidate[candidate_features].replace([np.inf, -np.inf], np.nan)
Xva = valid_candidate[candidate_features].replace([np.inf, -np.inf], np.nan)
ytr = train_candidate["target"].astype(np.float32)
yva = valid_candidate["target"].astype(np.float32)

# ============================================================
# 3. 训练候选模型并比较基准
# ============================================================
model_candidate = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1600,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_candidate.fit(Xtr, ytr)

pred_candidate = np.maximum(model_candidate.predict(Xva), 0)

# 基准模型使用之前保存的验证预测
if "pred_long" in globals() and len(pred_long) == len(yva):
    pred_base = np.maximum(np.asarray(pred_long), 0)
elif "valid_pred" in globals() and len(valid_pred) == len(yva):
    pred_base = np.maximum(np.asarray(valid_pred), 0)
else:
    pred_base = None

def report_metrics(name, y, pred):
    mae = np.mean(np.abs(y - pred))
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    return {"model": name, "MAE": mae, "RMSE": rmse}

results = [
    report_metrics("候选特征模型", yva.to_numpy(), pred_candidate)
]

if pred_base is not None:
    results.insert(
        0, report_metrics("现有基准预测", yva.to_numpy(), pred_base)
    )

results_df = pd.DataFrame(results)
print("\n========== 验证集指标 ==========")
print(results_df.to_string(index=False))

# ============================================================
# 4. 特征重要性
# ============================================================
importance_df = pd.DataFrame({
    "feature": candidate_features,
    "gain": model_candidate.booster_.feature_importance(importance_type="gain"),
    "split": model_candidate.booster_.feature_importance(importance_type="split")
}).sort_values("gain", ascending=False)

print("\n========== Gain 排名前 20 ==========")
print(importance_df.head(20).to_string(index=False))

# ============================================================
# 5. 按需求稀疏程度检查误差
# ============================================================
if "zero_ratio_56" in valid_candidate.columns:
    diagnostics = valid_candidate[["zero_ratio_56", "target"]].copy()
    diagnostics["pred"] = pred_candidate
    diagnostics["abs_error"] = np.abs(
        diagnostics["target"] - diagnostics["pred"]
    )
    diagnostics["demand_group"] = pd.cut(
        diagnostics["zero_ratio_56"],
        bins=[-0.001, 0.25, 0.75, 1.001],
        labels=["较连续", "间歇性", "高度稀疏"]
    )
    group_report = diagnostics.groupby(
        "demand_group", observed=False
    ).agg(
        rows=("target", "size"),
        actual_mean=("target", "mean"),
        pred_mean=("pred", "mean"),
        MAE=("abs_error", "mean")
    ).reset_index()

    print("\n========== 按需求稀疏程度的误差 ==========")
    print(group_report.to_string(index=False))

print("\n本轮新增特征数：", len(new_features))
print("新增特征：", new_features)
print("完成：候选模型已训练，基准与候选模型指标已输出。")


In [ ]:
from pathlib import Path

print("========== 已有模型和实验文件 ==========")
for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    if root.exists():
        for pattern in [
            "model_11.pkl",
            "model_long_exp.joblib",
            "m5_price_model.txt",
            "sales_train_evaluation.csv",
            "sales_train_validation.csv",
            "calendar.csv",
            "sell_prices.csv",
            "sample_submission.csv"
        ]:
            for p in root.rglob(pattern):
                print(p, f"({p.stat().st_size / 1024:.1f} KB)")

print("\n========== 当前内存中的相关对象 ==========")
for name in [
    "sales", "calendar", "train_df", "valid_df",
    "model_long_exp", "add_candidate_features",
    "prices", "sample_submission"
]:
    print(name, "存在" if name in globals() else "不存在")

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import lightgbm as lgb

WORK = Path("/kaggle/working")
DATA = Path("/kaggle/input/competitions/m5-forecasting-accuracy")

# 1. 加载已有的 16 特征实验模型
model_path = WORK / "m5_handover" / "model_long_exp.joblib"
model_long_exp = joblib.load(model_path)

print("========== 已恢复模型 ==========")
print("模型类型：", type(model_long_exp).__name__)

# 2. 读取模型实际使用的特征
if hasattr(model_long_exp, "booster_"):
    feature_names = model_long_exp.booster_.feature_name()
elif hasattr(model_long_exp, "feature_name"):
    feature_names = model_long_exp.feature_name()
else:
    feature_names = None

if feature_names is not None:
    print("模型特征数：", len(feature_names))
    print("模型特征列表：")
    print(feature_names)
else:
    print("未能从该模型对象直接读取特征名")

# 3. 检查官方提交模板
sample_submission = pd.read_csv(DATA / "sample_submission.csv")
print("\n========== 官方提交模板 ==========")
print("形状：", sample_submission.shape)
print("列名：", sample_submission.columns.tolist())
print("前 3 行：")
print(sample_submission.head(3))

# 4. 检查此前保存的实验文件
print("\n========== 已有实验文件 ==========")
for p in [
    WORK / "m5_checkpoints" / "long_features_aligned_5000.csv",
    WORK / "m5_checkpoints" / "long_sales_model_comparison.csv",
    WORK / "m5_checkpoints" / "long_sales_feature_importance.csv",
    WORK / "long_sales_features_5000.csv",
]:
    print(f"{p.name}: {'存在' if p.exists() else '不存在'}")

print("\n检查完成：尚未开始训练，也未生成或覆盖提交文件。")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/kaggle/input/competitions/m5-forecasting-accuracy")

sales = pd.read_csv(DATA / "sales_train_evaluation.csv")
calendar = pd.read_csv(DATA / "calendar.csv")
prices = pd.read_csv(DATA / "sell_prices.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

day_cols = [c for c in sales.columns if c.startswith("d_")]

print("销量表：", sales.shape)
print("日销量列数：", len(day_cols))
print("日历表：", calendar.shape)
print("价格表：", prices.shape)
print("提交模板：", sample_submission.shape)
print("商品序列数：", sales["id"].nunique())
print("销量历史范围：", day_cols[0], "到", day_cols[-1])
print("提交模板 ID 示例：", sample_submission["id"].head(3).tolist())

In [ ]:
import gc
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
OUT = Path("/kaggle/working/m5_36_model")
OUT.mkdir(exist_ok=True)

sales_df = pd.read_csv(DATA / "sales_train_evaluation.csv")
calendar = pd.read_csv(DATA / "calendar.csv")
prices = pd.read_csv(DATA / "sell_prices.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

day_cols = [f"d_{i}" for i in range(1, 1942)]
sales = sales_df[day_cols].to_numpy(dtype=np.float32)

meta = sales_df[
    ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
].copy()
meta["series_idx"] = np.arange(len(meta))

calendar["day_num"] = (
    calendar["d"].str.replace("d_", "", regex=False).astype(int)
)
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday_num"] = calendar["date"].dt.dayofweek
calendar["day_of_month"] = calendar["date"].dt.day
calendar["is_weekend"] = (calendar["weekday_num"] >= 5).astype(np.int8)

event_cols = [
    c for c in ["event_name_1", "event_name_2"]
    if c in calendar.columns
]
calendar["is_event"] = (
    calendar[event_cols].notna().any(axis=1).astype(np.int8)
    if event_cols else 0
)
calendar["near_event_3d"] = (
    calendar["is_event"]
    .rolling(7, center=True, min_periods=1)
    .max()
    .astype(np.int8)
)

cal_by_day = calendar.set_index("day_num").sort_index()

print("销量矩阵：", sales.shape)
print("商品序列数：", len(meta))
print("日历索引：", cal_by_day.shape)
print("价格表：", prices.shape)
print("提交模板：", sample_submission.shape)
print("数据准备完成")

In [ ]:
# ============================================================
# M5 | 候选特征 LightGBM：训练 + 保存 + 全量预测 + 提交验收
# ============================================================
import gc
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

# ---------- 0. 路径与参数 ----------
DATA = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
OUT = Path("/kaggle/working")
SAVE = OUT / "m5_36_model"
SAVE.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SERIES_TRAIN = 5000
N_ESTIMATORS = 1600

# ---------- 1. 读取数据 ----------
print("1/7 读取数据……")
sales_df = pd.read_csv(DATA / "sales_train_evaluation.csv")
calendar = pd.read_csv(DATA / "calendar.csv")
prices = pd.read_csv(DATA / "sell_prices.csv")
template = pd.read_csv(DATA / "sample_submission.csv")

day_cols = [f"d_{i}" for i in range(1, 1942)]
sales = sales_df[day_cols].to_numpy(dtype=np.float32)

meta = sales_df[
    ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
].copy()

n_series, n_days = sales.shape
assert n_series == 30490 and n_days == 1941
assert template.shape == (60980, 29)

# ---------- 2. 日历与价格索引 ----------
print("2/7 建立日历和价格索引……")
calendar["day_num"] = (
    calendar["d"].str.replace("d_", "", regex=False).astype(int)
)
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["weekday_num"] = calendar["date"].dt.dayofweek
calendar["day_of_month"] = calendar["date"].dt.day
calendar["is_weekend"] = (calendar["weekday_num"] >= 5).astype(np.int8)

event_cols = [
    c for c in ["event_name_1", "event_name_2"]
    if c in calendar.columns
]
calendar["is_event"] = (
    calendar[event_cols].notna().any(axis=1).astype(np.int8)
    if event_cols else 0
)
calendar["near_event_3d"] = (
    calendar["is_event"]
    .rolling(7, center=True, min_periods=1)
    .max()
    .astype(np.int8)
)

for c in ["snap_CA", "snap_TX", "snap_WI"]:
    if c not in calendar.columns:
        calendar[c] = 0

cal = calendar.set_index("day_num").sort_index()

# MultiIndex 只建立一次，避免每天扫描和合并整张价格表。
price_series = prices.set_index(
    ["item_id", "store_id", "wm_yr_wk"]
)["sell_price"].sort_index()

item_ids = meta["item_id"].to_numpy()
store_ids = meta["store_id"].to_numpy()
states = meta["state_id"].to_numpy()

def get_prices(day_num, series_indices):
    """按商品、门店、周直接索引价格；缺失价格记为 0。"""
    week = cal.loc[day_num, "wm_yr_wk"]
    idx = pd.MultiIndex.from_arrays([
        item_ids[series_indices],
        store_ids[series_indices],
        np.repeat(week, len(series_indices))
    ], names=["item_id", "store_id", "wm_yr_wk"])

    result = price_series.reindex(idx).to_numpy(dtype=np.float32)
    return np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)

# ---------- 3. 定义 36 个候选特征 ----------
BASE_FEATURES = [
    "weekday", "month", "year", "is_event", "snap", "sell_price",
    "lag_28", "lag_56", "rolling_mean_7", "rolling_mean_28",
    "lag_84", "mean_56", "zero_ratio_56", "nonzero_mean_56",
    "mean_28", "recent_vs_long_mean"
]

NEW_FEATURES = [
    "lag_364",
    "hist_mean_7", "hist_std_7", "hist_zero_ratio_7",
    "hist_mean_28", "hist_std_28", "hist_zero_ratio_28",
    "hist_mean_56", "hist_std_56", "hist_zero_ratio_56",
    "hist_mean_364", "hist_std_364", "hist_zero_ratio_364",
    "weekday_num", "day_of_month", "is_weekend",
    "near_event_3d", "price_log1p", "price_missing_new",
    "hist_mean_ratio_28_364"
]
ALL_FEATURES = BASE_FEATURES + NEW_FEATURES

all_indices = np.arange(n_series)

def build_features(day_num, history, indices):
    """
    day_num：目标日编号。
    history：目标日前已经观察到的销量，形状为商品数 × 历史天数。
    """
    r = cal.loc[day_num]
    n = history.shape[0]
    h = history.shape[1]

    def lag(k):
        if h < k:
            return np.zeros(n, dtype=np.float32)
        return history[:, h-k]

    def window(k):
        return history[:, max(0, h-k):h]

    def mean(k):
        return window(k).mean(axis=1).astype(np.float32)

    def std(k):
        return window(k).std(axis=1).astype(np.float32)

    def zero_ratio(k):
        return (window(k) == 0).mean(axis=1).astype(np.float32)

    def nonzero_mean(k):
        x = window(k)
        nz = x > 0
        count = nz.sum(axis=1)
        return np.divide(
            (x * nz).sum(axis=1),
            count,
            out=np.zeros(n, dtype=np.float32),
            where=count > 0
        )

    p = get_prices(day_num, indices)
    long_mean = mean(364)
    short_mean = mean(28)

    snap_col = np.where(
        states[indices] == "CA", "snap_CA",
        np.where(states[indices] == "TX", "snap_TX", "snap_WI")
    )
    snap = np.array(
        [r[c] for c in snap_col], dtype=np.float32
    )

    features = {
        "weekday": np.full(n, r["weekday_num"], dtype=np.float32),
        "month": np.full(n, r["date"].month, dtype=np.float32),
        "year": np.full(n, r["date"].year, dtype=np.float32),
        "is_event": np.full(n, r["is_event"], dtype=np.float32),
        "snap": snap,
        "sell_price": p,
        "lag_28": lag(28),
        "lag_56": lag(56),
        "rolling_mean_7": mean(7),
        "rolling_mean_28": short_mean,
        "lag_84": lag(84),
        "mean_56": mean(56),
        "zero_ratio_56": zero_ratio(56),
        "nonzero_mean_56": nonzero_mean(56),
        "mean_28": short_mean,
        "recent_vs_long_mean": short_mean / (long_mean + 1e-3),
        "lag_364": lag(364),
        "hist_mean_7": mean(7),
        "hist_std_7": std(7),
        "hist_zero_ratio_7": zero_ratio(7),
        "hist_mean_28": short_mean,
        "hist_std_28": std(28),
        "hist_zero_ratio_28": zero_ratio(28),
        "hist_mean_56": mean(56),
        "hist_std_56": std(56),
        "hist_zero_ratio_56": zero_ratio(56),
        "hist_mean_364": long_mean,
        "hist_std_364": std(364),
        "hist_zero_ratio_364": zero_ratio(364),
        "weekday_num": np.full(n, r["weekday_num"], dtype=np.float32),
        "day_of_month": np.full(n, r["day_of_month"], dtype=np.float32),
        "is_weekend": np.full(n, r["is_weekend"], dtype=np.float32),
        "near_event_3d": np.full(n, r["near_event_3d"], dtype=np.float32),
        "price_log1p": np.log1p(p),
        "price_missing_new": (p <= 0).astype(np.float32),
        "hist_mean_ratio_28_364": short_mean / (long_mean + 1e-3)
    }

    return pd.DataFrame(features, columns=ALL_FEATURES).replace(
        [np.inf, -np.inf], 0
    ).fillna(0)

# ---------- 4. 构造训练样本 ----------
print("3/7 构造训练样本……")
rng = np.random.default_rng(SEED)
train_indices = np.sort(
    rng.choice(n_series, size=min(N_SERIES_TRAIN, n_series), replace=False)
)

# 训练标签最晚到 d_1913，避免使用 validation 预测窗口的标签。
target_days = list(range(365, 1914, 7))
X_parts = []
y_parts = []

for j, d in enumerate(target_days, 1):
    hist = sales[train_indices, :d-1]
    Xd = build_features(d, hist, train_indices)
    X_parts.append(Xd)
    y_parts.append(sales[train_indices, d-1])

    if j % 25 == 0 or j == len(target_days):
        print(f"训练样本构造：{j}/{len(target_days)}")

X_train = pd.concat(X_parts, ignore_index=True)
y_train = np.concatenate(y_parts).astype(np.float32)

del X_parts, y_parts
gc.collect()

FEATURES_USED = [
    c for c in ALL_FEATURES
    if X_train[c].nunique(dropna=False) > 1
]
X_train = X_train[FEATURES_USED]

print("训练矩阵：", X_train.shape)
print("实际特征数：", len(FEATURES_USED))

# ---------- 5. 训练并立即保存 ----------
print("4/7 训练 LightGBM……")
model_candidate = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=N_ESTIMATORS,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)
model_candidate.fit(X_train, y_train)

MODEL_PATH = SAVE / "model_candidate_36.joblib"
FEATURE_PATH = SAVE / "feature_list.json"
IMPORTANCE_PATH = SAVE / "feature_importance.csv"

joblib.dump(model_candidate, MODEL_PATH)

with open(FEATURE_PATH, "w", encoding="utf-8") as f:
    json.dump(FEATURES_USED, f, ensure_ascii=False, indent=2)

importance = pd.DataFrame({
    "feature": FEATURES_USED,
    "gain": model_candidate.booster_.feature_importance(
        importance_type="gain"
    ),
    "split": model_candidate.booster_.feature_importance(
        importance_type="split"
    )
}).sort_values("gain", ascending=False)

importance.to_csv(IMPORTANCE_PATH, index=False)

# 保存后立即检查模型可以重新加载。
test_model = joblib.load(MODEL_PATH)
assert len(test_model.booster_.feature_name()) == len(FEATURES_USED)
del test_model, X_train, y_train
gc.collect()

print("模型已保存并重新加载验证通过：", MODEL_PATH)
print("特征清单：", FEATURE_PATH)

# ---------- 6. 递归预测两个窗口 ----------
def forecast_28(history, start_day):
    """
    history 包含 start_day 前一天及之前的真实销量。
    未来预测逐日递归，将预测值加入历史。
    """
    hist = history.copy()
    result = np.zeros((n_series, 28), dtype=np.float32)

    for step in range(28):
        d = start_day + step
        Xd = build_features(d, hist, all_indices)
        Xd = Xd[FEATURES_USED]

        pred = np.maximum(
            model_candidate.predict(Xd), 0
        ).astype(np.float32)

        result[:, step] = pred
        hist = np.column_stack((hist, pred))

        print(f"预测进度：d_{d}，第 {step+1}/28 天")

    return result

print("5/7 预测 validation：d_1914–d_1941……")
pred_validation = forecast_28(sales[:, :1913], 1914)

print("6/7 预测 evaluation：d_1942–d_1969……")
pred_evaluation = forecast_28(sales, 1942)

# ---------- 7. 组装、保存并验收提交文件 ----------
print("7/7 生成并验证 submission.csv……")
submission = template.copy()
forecast_cols = [f"F{i}" for i in range(1, 29)]

# 按官方模板 ID 顺序匹配，不改变行顺序。
id_to_idx = {sid: i for i, sid in enumerate(meta["id"])}
ids = submission["id"].astype(str).to_numpy()

for j, col in enumerate(forecast_cols):
    vals = np.zeros(len(submission), dtype=np.float32)

    for row_i, sid in enumerate(ids):
        if sid.endswith("_validation"):
            base_id = sid[:-len("_validation")]
            series_i = id_to_idx.get(base_id)
            if series_i is not None:
                vals[row_i] = pred_validation[series_i, j]

        elif sid.endswith("_evaluation"):
            base_id = sid[:-len("_evaluation")]
            series_i = id_to_idx.get(base_id)
            if series_i is not None:
                vals[row_i] = pred_evaluation[series_i, j]

    submission[col] = np.rint(np.maximum(vals, 0)).astype(np.int32)

SUBMISSION_PATH = OUT / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

# 验收文件形状、列名、ID 顺序、缺失值和非负性。
check = pd.read_csv(SUBMISSION_PATH)

assert check.shape == template.shape == (60980, 29)
assert check.columns.tolist() == template.columns.tolist()
assert check["id"].equals(template["id"])
assert check[forecast_cols].notna().all().all()
assert (check[forecast_cols].to_numpy() >= 0).all()

summary = {
    "model_path": str(MODEL_PATH),
    "feature_list_path": str(FEATURE_PATH),
    "importance_path": str(IMPORTANCE_PATH),
    "submission_path": str(SUBMISSION_PATH),
    "actual_feature_count": len(FEATURES_USED),
    "training_series_count": len(train_indices),
    "training_target_days": len(target_days),
    "submission_shape": list(check.shape),
    "submission_validated": True
}

with open(SAVE / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n========== 全部完成 ==========")
print("模型：", MODEL_PATH)
print("提交文件：", SUBMISSION_PATH)
print("提交形状：", check.shape)
print("实际特征数：", len(FEATURES_USED))
print("模型文件大小：", round(MODEL_PATH.stat().st_size / 1024**2, 2), "MB")
print("预测值范围：", int(check[forecast_cols].to_numpy().min()),
      "至", int(check[forecast_cols].to_numpy().max()))
print("验收通过：submission.csv 已生成，模型及特征清单已保存。")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# 1. 检查提交文件
sub_path = Path("/kaggle/working/submission.csv")
sub = pd.read_csv(sub_path)

pred_cols = [f"F{i}" for i in range(1, 29)]
pred = sub[pred_cols].to_numpy(dtype=np.float64)

print("========== 提交文件检查 ==========")
print("形状：", sub.shape)
print("预测值最小值：", pred.min())
print("预测值最大值：", pred.max())
print("非零预测值数量：", np.count_nonzero(pred))
print("非零预测占比：", np.mean(pred != 0))
print("NaN 数量：", np.isnan(pred).sum())
print("前5行预测：")
print(sub[["id"] + pred_cols[:5]].head().to_string(index=False))

# 2. 检查模型是否真的学到了非零输出
model_path = Path(
    "/kaggle/working/m5_36_model/model_candidate_36.joblib"
)
model = joblib.load(model_path)

print("\n========== 模型检查 ==========")
print("模型类型：", type(model).__name__)
print("模型特征数：", getattr(model, "n_features_in_", "未知"))

if hasattr(model, "feature_importances_"):
    imp = model.feature_importances_
    print("特征重要性总和：", imp.sum())
    print("非零重要性特征数：", np.count_nonzero(imp))
    print("重要性最高的10个特征索引：",
          np.argsort(imp)[-10:][::-1])

# 3. 检查真实销量是否正常
data_path = (
    "/kaggle/input/competitions/m5-forecasting-accuracy/"
    "sales_train_evaluation.csv"
)
sales_df = pd.read_csv(data_path)
sales_cols = [c for c in sales_df.columns if c.startswith("d_")]
sales = sales_df[sales_cols].to_numpy(dtype=np.float32)

print("\n========== 真实销量检查 ==========")
print("历史销量范围：", sales.min(), "至", sales.max())
print("历史销量非零占比：", np.mean(sales != 0))
print("最后28天真实销量均值：", sales[:, -28:].mean())

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/kaggle/working/m5_36_model")
model = joblib.load(ROOT / "model_candidate_36.joblib")

with open(ROOT / "feature_list.json", "r") as f:
    feature_names = json.load(f)

print("特征清单类型：", type(feature_names).__name__)
if isinstance(feature_names, dict):
    print("清单键：", list(feature_names.keys()))
    # 兼容常见的 JSON 保存格式
    for k in ["features", "feature_names", "candidate_features"]:
        if k in feature_names:
            feature_names = feature_names[k]
            break

print("特征数：", len(feature_names))
print("模型特征数：", model.n_features_in_)

# 用历史训练数据构造一个最简单的诊断输入
# 每个特征先用 0 填充，确保输入维度正确
X_zero = pd.DataFrame(
    np.zeros((5, len(feature_names)), dtype=np.float32),
    columns=feature_names
)

p_zero = model.predict(X_zero)

print("\n【全零特征输入】")
print("预测值：", p_zero)
print("范围：", float(np.min(p_zero)), "至", float(np.max(p_zero)))

# 再用训练过程中保存的特征重要性，确认模型树正常
print("\n【模型结构】")
print("树数量：", model.n_estimators_)
print("非零特征重要性：", int(np.count_nonzero(model.feature_importances_)))
print("前10个特征：")
print(pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).head(10).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
from pathlib import Path

ROOT = Path("/kaggle/working/m5_36_model")
DATA = "/kaggle/input/competitions/m5-forecasting-accuracy"

model = joblib.load(ROOT / "model_candidate_36.joblib")
with open(ROOT / "feature_list.json", "r") as f:
    features = json.load(f)

sales_df = pd.read_csv(f"{DATA}/sales_train_evaluation.csv")
sales_cols = [f"d_{i}" for i in range(1, 1942)]
sales = sales_df[sales_cols].to_numpy(dtype=np.float32)

# 取前100条商品序列，以 d_1913 之前的真实历史构造基础滞后特征
hist = sales[:100, :1913]
n = len(hist)

X = pd.DataFrame(0.0, index=range(n), columns=features)

# 填入能直接从历史销量计算的特征
for lag in [28, 56, 84, 364]:
    col = f"lag_{lag}"
    if col in X.columns:
        X[col] = hist[:, -lag]

for window, names in [
    (7, ["rolling_mean_7", "hist_mean_7", "hist_std_7",
         "hist_zero_ratio_7"]),
    (28, ["rolling_mean_28", "hist_mean_28", "hist_std_28",
          "hist_zero_ratio_28"]),
    (56, ["hist_mean_56", "hist_std_56", "hist_zero_ratio_56"]),
    (364, ["hist_mean_364", "hist_std_364", "hist_zero_ratio_364"])
]:
    recent = hist[:, -window:]
    for col in names:
        if col in X.columns:
            if "std" in col:
                X[col] = recent.std(axis=1)
            elif "zero_ratio" in col:
                X[col] = (recent == 0).mean(axis=1)
            else:
                X[col] = recent.mean(axis=1)

if "mean_28" in X.columns:
    X["mean_28"] = hist[:, -28:].mean(axis=1)
if "mean_56" in X.columns:
    X["mean_56"] = hist[:, -56:].mean(axis=1)
if "zero_ratio_56" in X.columns:
    X["zero_ratio_56"] = (hist[:, -56:] == 0).mean(axis=1)
if "nonzero_mean_56" in X.columns:
    z = hist[:, -56:]
    X["nonzero_mean_56"] = np.divide(
        (z * (z != 0)).sum(axis=1),
        (z != 0).sum(axis=1),
        out=np.zeros(n, dtype=np.float32),
        where=(z != 0).sum(axis=1) > 0
    )
if "mean_56" in X.columns and "mean_364" in X.columns:
    X["mean_56"] = hist[:, -56:].mean(axis=1)
    X["mean_364"] = hist[:, -364:].mean(axis=1)
if "recent_vs_long_mean" in X.columns:
    X["recent_vs_long_mean"] = (
        hist[:, -28:].mean(axis=1) /
        (hist[:, -364:].mean(axis=1) + 1e-6)
    )
if "hist_mean_ratio_28_364" in X.columns:
    X["hist_mean_ratio_28_364"] = (
        hist[:, -28:].mean(axis=1) /
        (hist[:, -364:].mean(axis=1) + 1e-6)
    )

pred = model.predict(X[features])

print("诊断输入形状：", X.shape)
print("真实历史销量均值：", float(hist.mean()))
print("模型预测范围：", float(pred.min()), "至", float(pred.max()))
print("模型预测均值：", float(pred.mean()))
print("前10个预测：", np.round(pred[:10], 3))

sub = pd.read_csv("/kaggle/working/submission.csv")
cols = [f"F{i}" for i in range(1, 29)]
print("\n提交文件预测均值：", float(sub[cols].to_numpy().mean()))
print("提交文件非零值数量：",
      int(np.count_nonzero(sub[cols].to_numpy())))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. 检查提交文件的 ID 和预测列
path = Path("/kaggle/working/submission.csv")
sub = pd.read_csv(path)

print("提交 ID 示例：")
print(sub["id"].head(3).tolist())
print("\n提交 ID 后缀统计：")
print(sub["id"].str.rsplit("_", n=1).str[-1].value_counts())

# 2. 检查销量数据中的 ID
data_path = (
    "/kaggle/input/competitions/m5-forecasting-accuracy/"
    "sales_train_evaluation.csv"
)
sales_meta = pd.read_csv(
    data_path,
    usecols=["id", "item_id", "store_id"]
)

print("\n销量数据 ID 示例：")
print(sales_meta["id"].head(3).tolist())

print("\n销量数据 ID 后缀统计：")
print(sales_meta["id"].str.rsplit("_", n=1).str[-1].value_counts())

# 3. 检查提交 ID 去掉后缀后，能否匹配销量数据
sub_base = sub["id"].str.replace(
    r"_(validation|evaluation)$", "", regex=True
)
sales_base = sales_meta["id"].str.replace(
    r"_(validation|evaluation)$", "", regex=True
)

print("\n提交基础 ID 匹配率：",
      sub_base.isin(set(sales_base)).mean())
print("提交基础 ID 唯一数：", sub_base.nunique())
print("销量基础 ID 唯一数：", sales_base.nunique())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working")
DATA = Path("/kaggle/input/competitions/m5-forecasting-accuracy")

# 读取官方提交模板
template = pd.read_csv(DATA / "sample_submission.csv")

# 使用当前 Notebook 已生成的预测数组
assert "pred_validation" in globals(), "找不到 pred_validation，请勿重新训练，先检查 Notebook 状态"
assert "pred_evaluation" in globals(), "找不到 pred_evaluation，请勿重新训练，先检查 Notebook 状态"

# 读取商品 ID，并统一去掉 _evaluation 后缀
sales_meta = pd.read_csv(
    DATA / "sales_train_evaluation.csv",
    usecols=["id"]
)
base_ids = sales_meta["id"].str.replace(
    r"_evaluation$", "", regex=True
)

id_to_idx = {sid: i for i, sid in enumerate(base_ids)}

assert len(id_to_idx) == 30490
assert pred_validation.shape == (30490, 28)
assert pred_evaluation.shape == (30490, 28)

# 按模板 ID 精确写入预测
submission = template.copy()
ids = submission["id"].astype(str)

for j in range(1, 29):
    col = f"F{j}"
    vals = np.zeros(len(submission), dtype=np.float32)

    validation_mask = ids.str.endswith("_validation").to_numpy()
    evaluation_mask = ids.str.endswith("_evaluation").to_numpy()

    # 去掉对应后缀，得到基础商品 ID
    validation_base = ids[validation_mask].str.replace(
        r"_validation$", "", regex=True
    )
    evaluation_base = ids[evaluation_mask].str.replace(
        r"_evaluation$", "", regex=True
    )

    validation_idx = validation_base.map(id_to_idx).to_numpy()
    evaluation_idx = evaluation_base.map(id_to_idx).to_numpy()

    assert pd.notna(validation_idx).all()
    assert pd.notna(evaluation_idx).all()

    vals[validation_mask] = pred_validation[
        validation_idx.astype(int), j - 1
    ]
    vals[evaluation_mask] = pred_evaluation[
        evaluation_idx.astype(int), j - 1
    ]

    # 保留非负浮点预测，不额外四舍五入
    submission[col] = np.maximum(vals, 0)

# 保存修复后的提交文件
path = OUT / "submission.csv"
submission.to_csv(path, index=False)

# 重新读取并严格验收
check = pd.read_csv(path)
forecast_cols = [f"F{i}" for i in range(1, 29)]
pred = check[forecast_cols].to_numpy(dtype=np.float64)

assert check.shape == (60980, 29)
assert check["id"].equals(template["id"])
assert np.isfinite(pred).all()
assert (pred >= 0).all()
assert np.count_nonzero(pred) > 0, "预测仍然全为 0，请停止提交"

print("========== 修复完成 ==========")
print("文件：", path)
print("形状：", check.shape)
print("预测最小值：", pred.min())
print("预测最大值：", pred.max())
print("预测均值：", pred.mean())
print("非零预测数量：", np.count_nonzero(pred))
print("非零预测占比：", np.mean(pred != 0))
print("\n前5行：")
print(check[["id"] + forecast_cols[:5]].head().to_string(index=False))
print("\n验收通过：ID 匹配、预测有限、预测非负、且并非全零。")